# **비정형 데이터 처리 Final Term Project**

### **한국 상장기업 사업보고서의 ESG 관련 표현이 KCGS ESG 등급과 어떤 연관성을 보이는지 검증**

**3조**: 202231193 김혜성 · 202136006 이동원 · 202431626 김지우 · 202431928 신지영  
**표본** : KCGS ESG 등급 보유 127개 상장기업 × 3개 회계연도 (2022–2024) = 381 firm-year  
**타이밍** : `esg_year = fiscal_year + 1` (KCGS 평가연도 t ↔ 직전 회계연도 t−1 사업보고서)

## **연구 질문**

> 사업보고서 텍스트에서 추출한 ESG 관련 어휘 강도가 KCGS ESG 등급과 통계적으로 연관되는가?  
> 이 연관성을 공시 장황함(cheap-talk)과 어떻게 구분할 수 있는가?  
→ 인과관계 식별이 아닌 공시 언어와 외부 평가 사이의 패턴을 데이터로 확인

## **목차**

| 단계 | 내용 |
|---|---|
| 0 | 환경 셋업 |
| 1 | 데이터 수집 (381 firm-year) |
| 2 | 전처리 — 형태소 분석기 · seed 보호 · 불용어 |
| 3 | Feature 생성 — TF-IDF · FastText · Cosine |
| 4 | Validity — Spearman · Mann-Whitney |
| 5 | 회귀 3종 — OLS · Ordered · Binary |
| 6 | 알파 분석 — Cheap-talk · Robustness |
| 7 | Cheap-talk Evidence Matrix |
| 8 | 종합 결론 · 한계 · Self-Check |

In [ ]:
"""
0. 환경 셋업
- 라이브러리 import
- 경로 설정 및 폴더 생성
- API key 로드 (환경변수)
- 시드 고정 (재현성)
- 라이브러리 버전 출력
"""
import os, re, sys, json, html, time, pickle, requests, warnings, zipfile
from io import BytesIO
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- 1) API key ---
load_dotenv()
API_KEY = os.getenv("OPENDART_API_KEY") or os.getenv("DART_API_KEY")
assert API_KEY is not None, (
    "OPENDART_API_KEY가 .env에 없습니다. "
    "프로젝트 루트에 .env 파일을 만들고 'OPENDART_API_KEY=...'를 추가하세요."
)
print(f"[ENV] API_KEY 로드 OK (앞 8자: {API_KEY[:8]}...)")

# --- 2) 경로 설정 ---
DATA_DIR   = "data"
OUTPUT_DIR = "outputs"
CORPUS_DIR = os.path.join(OUTPUT_DIR, "corpus")
ZIPS_DIR   = os.path.join(OUTPUT_DIR, "doc_zips")
SEED       = 42

for d in [OUTPUT_DIR, CORPUS_DIR, ZIPS_DIR]:
    os.makedirs(d, exist_ok=True)

master_csv = os.path.join(DATA_DIR, "company_master.csv")
assert os.path.exists(master_csv), "data/company_master.csv 없음"

print(f"\n[PATH]")
print(f"  DATA_DIR   : {DATA_DIR}/")
print(f"  OUTPUT_DIR : {OUTPUT_DIR}/")
print(f"  CORPUS_DIR : {CORPUS_DIR}/")
print(f"  ZIPS_DIR   : {ZIPS_DIR}/")
print(f"  master_csv : {master_csv} ✓")

# --- 3) 시드 고정 ---
np.random.seed(SEED)
print(f"\n[SEED] {SEED}")

# --- 4) 버전 출력 ---
print(f"\n[LIB VERSIONS]")
print(f"  python   : {sys.version.split()[0]}")
print(f"  pandas   : {pd.__version__}")
print(f"  numpy    : {np.__version__}")
print(f"  requests : {requests.__version__}")

# --- 5) outputs/ 현황 ---
existing = os.listdir(OUTPUT_DIR)
print(f"\n[OUTPUTS 현황] {len(existing)}개 파일/폴더")
if existing:
    for f in sorted(existing)[:10]:
        print(f"  - {f}")
    print(f"  (빈 환경부터 시작하려면 outputs/ 를 비우세요)")
else:
    print(f"  ✓ 비어있음")

[ENV] API_KEY 로드 OK (앞 8자: 00814516...)

[PATH]
  DATA_DIR   : data/
  OUTPUT_DIR : outputs/
  CORPUS_DIR : outputs\corpus/
  ZIPS_DIR   : outputs\doc_zips/
  master_csv : data\company_master.csv ✓

[SEED] 42

[LIB VERSIONS]
  python   : 3.12.13
  pandas   : 3.0.1
  numpy    : 2.4.2
  requests : 2.32.5

[OUTPUTS 현황] 2개 파일/폴더
  - corpus
  - doc_zips
  (빈 환경부터 시작하려면 outputs/ 를 비우세요)


---

# **1. 데이터 수집 (381 firm-year)**

## **1-1. stock_code · timing 설계**

### **Decision Box ① — stock_code 정규화**

- DART API는 6자리 문자열 stock_code 요구
- CSV를 정수로 읽으면 `005930 → 5930` — corp_code 매핑 실패로 수집 불가
- CSV 로드 직후 `dtype={"stock_code": str}` + `.str.zfill(6)` 강제 적용

### **Decision Box ② — timing 정렬**

- esg_year = fiscal_year + 1 (KCGS 평가연도 t ↔ 직전 회계연도 t−1 보고서)
- CSV의 `esg_year` 컬럼이 `fiscal_year`와 동일값 — CSV 원본은 변형하지 않고 DART 검색 범위를 `fiscal_year + 1`년으로 지정

### **Decision Box ③ — corp_code 매핑 방식**

- 전체 상장기업 stock_code ↔ corp_code 매핑을 1회 호출로 확보
- Checkpoint 패턴으로 재실행 시 다운로드 skip

In [ ]:
"""
1-1. company_master.csv 로드 + 식별자 진단 + corp_code 매핑

Decision Box ②③④⑤ 구현
- stock_code zfill(6) 강제
- timing 진단 (esg_year vs fiscal_year)
- corpCode.xml로 corp_code 매핑
"""
import xml.etree.ElementTree as ET

# ── company_master 로드 ───────────────────────────────────────────
cm = pd.read_csv(master_csv, dtype={"stock_code": str})
cm["stock_code"] = cm["stock_code"].str.zfill(6)

print("[company_master.csv 로드]")
print(f"  shape  : {cm.shape}")
print(f"  columns: {list(cm.columns)}")

# ── 식별자 진단 ─────────────────────────────────────────────────────
n_name  = cm["company_name"].nunique()
n_stock = cm["stock_code"].nunique()
print(f"\n[식별자 진단]")
print(f"  전체 행수           : {len(cm)}")
print(f"  unique company_name : {n_name}")
print(f"  unique stock_code   : {n_stock}")
print(f"  이름-코드 차이      : {n_name - n_stock}개 → stock_code × fiscal_year 키 사용")

# ── stock_code 길이 검증 (Decision Box ①) ────────────────────────
assert all(cm["stock_code"].str.len() == 6), "zfill 실패"
print(f"\n[stock_code zfill(6) 검증] 전행 6자리 ✓")

# ── fiscal_year 분포 ──────────────────────────────────────────────
print(f"\n[fiscal_year 분포]")
print(cm["fiscal_year"].value_counts().sort_index().to_string())

# ── timing 진단 (Decision Box ②) ─────────────────────────────────
print(f"\n[timing 진단]")
diff = (cm["esg_year"] - cm["fiscal_year"]).value_counts().sort_index()
print(f"  (esg_year - fiscal_year) 분포:")
print(diff.to_string())
cm["search_year"] = cm["fiscal_year"] + 1
print(f"\n  → search_year = fiscal_year + 1 적용")
print(cm[["fiscal_year","esg_year","search_year"]].drop_duplicates()
      .sort_values("fiscal_year").to_string(index=False))

# ── ESG 등급 컬럼 확인 ────────────────────────────────────────────
grade_cols = [c for c in cm.columns if "grade" in c.lower()]
print(f"\n[ESG 등급 컬럼]")
for col in grade_cols:
    print(f"  {col}: {sorted(cm[col].dropna().unique().tolist())}")

# ── corp_code 매핑 (Decision Box ③) ──────────────────────────────
print(f"\n[corp_code 결측 현황]")
n_missing = cm["corp_code"].isna().sum()
print(f"  결측: {n_missing} / {len(cm)}")

corpcode_path = os.path.join(OUTPUT_DIR, "corpCode.xml")
if os.path.exists(corpcode_path):
    print(f"\n  [Checkpoint] corpCode.xml 존재 — 다운로드 skip")
else:
    print(f"\n  [다운로드 시작]")
    res = requests.get(
        "https://opendart.fss.or.kr/api/corpCode.xml",
        params={"crtfc_key": API_KEY}, timeout=60,
    )
    res.raise_for_status()
    with zipfile.ZipFile(BytesIO(res.content)) as zf:
        xml_name  = [n for n in zf.namelist() if n.endswith(".xml")][0]
        xml_bytes = zf.read(xml_name)
    with open(corpcode_path, "wb") as f:
        f.write(xml_bytes)
    print(f"  저장: {corpcode_path} ({len(xml_bytes)/1e6:.1f}MB)")

tree   = ET.parse(corpcode_path)
root   = tree.getroot()
listed = pd.DataFrame([
    {"corp_code":  (c.findtext("corp_code")  or "").strip(),
     "stock_code": (c.findtext("stock_code") or "").strip()}
    for c in root.findall(".//list")
])
listed = listed[listed["stock_code"].str.len() > 0].copy()
listed["stock_code"] = listed["stock_code"].str.zfill(6)
stock_to_corp = dict(zip(listed["stock_code"], listed["corp_code"]))

cm["corp_code"] = cm["stock_code"].map(stock_to_corp)
cm["corp_code"] = (cm["corp_code"].fillna("").str.zfill(8)
                   .replace("00000000", ""))

n_mapped = (cm["corp_code"].str.len() == 8).sum()
print(f"\n[corp_code 매핑 결과]")
print(f"  성공: {n_mapped} / {len(cm)}")
if n_mapped < len(cm):
    fail = cm[cm["corp_code"].str.len() != 8]
    print(f"  실패 행:")
    print(fail[["company_name","stock_code","fiscal_year"]].drop_duplicates().to_string(index=False))

# ── Sanity check ──────────────────────────────────────────────────
sam = cm[cm["stock_code"] == "005930"][["company_name","stock_code","corp_code"]].drop_duplicates()
print(f"\n[Sanity check — 삼성전자]")
print(sam.to_string(index=False))
print(f"  → 정답 corp_code: 00126380")

META_INPUT_DF = cm.copy()
print(f"\n[META_INPUT_DF 저장] shape={META_INPUT_DF.shape}")

[company_master.csv 로드]
  shape  : (381, 19)
  columns: ['team_id', 'role', 'company_name', 'corp_code', 'stock_code', 'industry', 'fiscal_year', 'report_code', 'rcept_no', 'esg_source', 'esg_grade', 'e_grade', 's_grade', 'g_grade', 'esg_year', 'is_required', 'replacement_of', 'replacement_reason', 'notes']

[식별자 진단]
  전체 행수           : 381
  unique company_name : 133
  unique stock_code   : 127
  이름-코드 차이      : 6개 → stock_code × fiscal_year 키 사용

[stock_code zfill(6) 검증] 전행 6자리 ✓

[fiscal_year 분포]
fiscal_year
2022    127
2023    127
2024    127

[timing 진단]
  (esg_year - fiscal_year) 분포:
0    381

  → search_year = fiscal_year + 1 적용
 fiscal_year  esg_year  search_year
        2022      2022         2023
        2023      2023         2024
        2024      2024         2025

[ESG 등급 컬럼]
  esg_grade: ['A', 'A+', 'B', 'B+', 'C', 'D']
  e_grade: ['A', 'A+', 'B', 'B+', 'C', 'D']
  s_grade: ['A', 'A+', 'B', 'B+', 'C', 'D']
  g_grade: ['A', 'A+', 'B', 'B+', 'C', 'D']

[corp_code 결측 현황]
  

**company_master.csv 진단 + corp_code 매핑 완료**

| 항목 | 값 | 판단 |
|---|---|---|
| 전체 행수 | 381 | 127개 × 3년 정상 |
| unique company_name | 133 | stock_code(127)보다 6개 많음 — 사명변경/분할 존재 확인 |
| unique stock_code | 127 | join 키로 사용 |
| stock_code 길이 | 전행 6자리 | zfill(6) 정상 작동 |
| esg_year − fiscal_year | 0 (381행 전체) | CSV는 동일값 → search_year = fiscal_year + 1 적용 |
| corp_code 매핑 | 381 / 381 성공 | corpCode.xml Checkpoint 활용 |

- ESG 등급 4개 차원(esg/e/s/g) 모두 D·C·B·B+·A·A+ 6개 등급 — 표본 최고 등급 A+
- CSV의 `corp_code` 컬럼은 381행 전체 결측 (의도적 공란) → corpCode.xml 매핑으로 정상 확보
- 삼성전자 sanity check 통과 (`005930 → 00126380`)

## **1-2. 수집 함수 설계**

DART 사업보고서 수집은 세 단계 함수로 구성한다.

### **Decision Box ④ — 사업보고서 검색 전략**

- `"사업보고서"` 포함이면 매칭, 아래 키워드 포함 시 제외
  - `"반기"` · `"분기"` — 반기·분기보고서 혼입 방지
  - `"연장"` · `"신고서"` — 제출기한 연장신고서 혼입 방지
- 정정공시 우선순위: 원본(0) > `[기재정정]`(1) > `[첨부정정]`(2), 동순위는 `rcept_dt` 오름차순

### **Decision Box ⑤ — 본 보고서 XML 선택**

- DART ZIP 내부에는 본 보고서 외 첨부서류 XML이 함께 들어있음
- `{rcept_no}.xml` 이름이 정확히 일치하면 본 보고서가 확실
- `{rcept_no}.xml` → 언더스코어 없는 XML → 가장 큰 XML

### **Decision Box ⑥ — ESG 섹션 추출 범위**

| 섹션 | 내용 | ESG 관련성 |
|---|---|---|
| II. 사업의 내용 | 주요 사업·환경·사회 활동 서술 | E·S 핵심 |
| IV. 이사의 경영진단 및 분석의견 | 경영진 ESG 방향성·성과 서술 | E·S·G 통합 |
| VI. 이사회 등 회사의 기관에 관한 사항 | 이사회 구성·감사위원회·내부통제 | G 핵심 |


**섹션 경계 탐지 방식**

- 대분류 로마숫자 TITLE(I·II·III…)만 경계로 사용 → 소제목은 경계 제외

In [ ]:
"""
1-2. 수집 함수 4종 정의

find_business_report_v2 : 사업보고서 rcept_no 검색 (Decision Box ④)
download_document_xml   : 원문 ZIP → 본 보고서 XML (Decision Box ⑤)
extract_esg_sections    : XML → II/IV/VI 섹션 텍스트 (Decision Box ⑥)
collect_one_firm_year   : 위 3종 묶음
"""

# ── 함수 1 ───────────────────────────────────────────────────────
def find_business_report_v2(corp_code, fiscal_year, api_key, verbose=False):
    """
    fiscal_year+1년 공시 사업보고서 검색.
    반기·분기·연장·신고서 제외, 정정 우선순위 적용.
    """
    search_year = fiscal_year + 1
    res = requests.get(
        "https://opendart.fss.or.kr/api/list.json",
        params={
            "crtfc_key":        api_key,
            "corp_code":        corp_code,
            "bgn_de":           f"{search_year}0101",
            "end_de":           f"{search_year}1231",
            "pblntf_detail_ty": "A001",
            "page_count":       "100",
        },
        timeout=20,
    )
    payload = res.json()
    if payload.get("status") != "000":
        if verbose:
            print(f"  status={payload.get('status')}: {payload.get('message')}")
        return None

    candidates = [
        r for r in payload.get("list", [])
        if "사업보고서" in r.get("report_nm", "")
        and "반기"   not in r.get("report_nm", "")
        and "분기"   not in r.get("report_nm", "")
        and "연장"   not in r.get("report_nm", "")
        and "신고서" not in r.get("report_nm", "")
    ]
    if not candidates:
        return None

    def priority(name):
        if "[첨부정정]" in name: return 2
        if "[기재정정]" in name: return 1
        return 0

    candidates.sort(key=lambda r: (priority(r["report_nm"]), r["rcept_dt"]))
    c = candidates[0]
    return {
        "rcept_no":     c["rcept_no"],
        "rcept_dt":     c["rcept_dt"],
        "report_nm":    c["report_nm"],
        "n_candidates": len(candidates),
    }


# ── 함수 2 ───────────────────────────────────────────────────────
def download_document_xml(rcept_no, api_key, output_dir, verbose=False):
    """
    ZIP 내 본 보고서 XML 선택 우선순위 (Decision Box ⑤):
    1. {rcept_no}.xml 정확히 일치
    2. 언더스코어 없는 XML 첫 번째
    3. 가장 큰 XML
    """
    zip_path = os.path.join(output_dir, "doc_zips", f"doc_{rcept_no}.zip")
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)

    if not os.path.exists(zip_path):
        res = requests.get(
            "https://opendart.fss.or.kr/api/document.xml",
            params={"crtfc_key": api_key, "rcept_no": rcept_no},
            timeout=60,
        )
        res.raise_for_status()
        if res.content[:2] != b"PK":
            raise ValueError(f"ZIP 아님: {res.content[:200]}")
        with open(zip_path, "wb") as f:
            f.write(res.content)

    with zipfile.ZipFile(zip_path) as zf:
        xml_names = [n for n in zf.namelist() if n.endswith(".xml")]
        if not xml_names:
            raise ValueError(f"ZIP 내 XML 없음: {zf.namelist()}")
        main = f"{rcept_no}.xml"
        if main in xml_names:
            chosen = main
        else:
            no_suffix = [n for n in xml_names if "_" not in n]
            chosen = no_suffix[0] if no_suffix else max(
                xml_names, key=lambda n: zf.getinfo(n).file_size
            )
        xml_bytes = zf.read(chosen)

    for enc in ("utf-8", "euc-kr", "cp949"):
        try:
            return xml_bytes.decode(enc), zip_path, chosen
        except UnicodeDecodeError:
            continue
    return xml_bytes.decode("utf-8", errors="ignore"), zip_path, chosen


# ── 함수 3 ───────────────────────────────────────────────────────
def extract_esg_sections(xml_text, verbose=False):
    """
    대상: II·IV·VI (Decision Box ⑥)
    경계: 대분류 로마숫자 TITLE만 사용 (소제목 제외 — 버그 수정)
    처리: TABLE 블록 제거 → P 태그 텍스트 추출 → 10자 미만 제거
    """
    ROMAN_PAT = re.compile(
        r"^(I{1,3}|IV|VI{0,3}|IX|X{0,3}(?:I{1,3}|IV|VI{0,3})?)[.\s]"
    )
    TARGET = {"II", "IV", "VI"}

    all_titles = [
        (m.start(), m.end(), re.sub(r"<[^>]+>", "", m.group(1)).strip())
        for m in re.finditer(r"<TITLE[^>]*>(.*?)</TITLE>", xml_text, re.DOTALL)
    ]
    roman_titles = [
        (s, e, t) for s, e, t in all_titles if ROMAN_PAT.match(t)
    ]

    results = {}
    for i, (start, end, text) in enumerate(roman_titles):
        roman_key = ROMAN_PAT.match(text).group(1)
        if roman_key not in TARGET:
            continue
        chunk_start = end
        chunk_end   = roman_titles[i+1][0] if i+1 < len(roman_titles) else len(xml_text)
        chunk = xml_text[chunk_start:chunk_end]
        chunk = re.sub(r"<TABLE[^>]*>.*?</TABLE>", " ", chunk, flags=re.DOTALL)
        paras = re.findall(r"<P[^>]*>(.*?)</P>", chunk, re.DOTALL)
        texts = [
            html.unescape(re.sub(r"<[^>]+>", "", p)).strip()
            for p in paras
            if len(html.unescape(re.sub(r"<[^>]+>", "", p)).strip()) >= 10
        ]
        if texts:
            results[roman_key] = "\n".join(texts)
    return results


# ── 함수 4 ───────────────────────────────────────────────────────
def collect_one_firm_year(row, api_key, output_dir, verbose=False):
    """단일 firm-year 수집 파이프라인."""
    stock_code  = row["stock_code"]
    corp_code   = row["corp_code"]
    fiscal_year = int(row["fiscal_year"])

    found = find_business_report_v2(corp_code, fiscal_year, api_key, verbose)
    if found is None:
        return {"status": "FAIL", "reason": "보고서_미발견",
                "stock_code": stock_code, "fiscal_year": fiscal_year}
    try:
        xml_text, zip_path, xml_name = download_document_xml(
            found["rcept_no"], api_key, output_dir, verbose
        )
    except Exception as e:
        return {"status": "FAIL", "reason": f"ZIP오류:{e}",
                "stock_code": stock_code, "fiscal_year": fiscal_year, **found}

    sections = extract_esg_sections(xml_text, verbose)
    if not sections:
        return {"status": "FAIL", "reason": "섹션_미추출",
                "stock_code": stock_code, "fiscal_year": fiscal_year, **found}

    combined = "\n\n".join(sections.values())
    return {
        "status":           "SUCCESS",
        "reason":           "",
        "stock_code":       stock_code,
        "fiscal_year":      fiscal_year,
        "rcept_no":         found["rcept_no"],
        "rcept_dt":         found["rcept_dt"],
        "report_nm":        found["report_nm"],
        "viewer_url":       f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={found['rcept_no']}",
        "text":             combined,
        "section_chars_II": len(sections.get("II", "")),
        "section_chars_IV": len(sections.get("IV", "")),
        "section_chars_VI": len(sections.get("VI", "")),
        "total_chars":      len(combined),
    }

print("[함수 정의 완료]")
print("  find_business_report_v2  ✓  (반기·분기·연장 제외, 정정 우선순위)")
print("  download_document_xml    ✓  (본 보고서 XML 우선순위 선택)")
print("  extract_esg_sections     ✓  (대분류 로마숫자 경계, TABLE 제거)")
print("  collect_one_firm_year    ✓  (3종 묶음, SUCCESS/FAIL 반환)")

[함수 정의 완료]
  find_business_report_v2  ✓  (반기·분기·연장 제외, 정정 우선순위)
  download_document_xml    ✓  (본 보고서 XML 우선순위 선택)
  extract_esg_sections     ✓  (대분류 로마숫자 경계, TABLE 제거)
  collect_one_firm_year    ✓  (3종 묶음, SUCCESS/FAIL 반환)


**수집 함수 4종 정의 완료**

| 함수 | 핵심 설계 결정 |
|---|---|
| `find_business_report_v2` | 반기·분기·연장·신고서 제외, 정정 우선순위(원본>기재>첨부) |
| `download_document_xml` | `{rcept_no}.xml` → 언더스코어 없는 XML → 가장 큰 XML 순 선택 |
| `extract_esg_sections` | 대분류 로마숫자 TITLE만 경계로 사용, TABLE 블록 제거, 10자 미만 제거 |
| `collect_one_firm_year` | 3종 묶음, 단계별 FAIL 사유 반환 |

## **1-3. Sanity Check — 삼성전자 FY2024**

전체 수집 전에 1건으로 파이프라인 전 과정을 검증한다.

**확인 항목**
- `find_business_report_v2`: 올바른 `rcept_no` · `report_nm` 반환 여부
- `download_document_xml`: ZIP 정상 다운로드, 본 보고서 XML 선택 여부
- `extract_esg_sections`: II·IV·VI 3개 섹션 모두 추출 여부 및 글자 수
- 추출 텍스트 앞 200자 직접 확인 — ESG 관련 표현 포함 여부
- 섹션별 글자 수가 0이면 함수 수정 후 재검증

In [22]:
"""
1-3. Sanity Check — 삼성전자 FY2024
전체 수집 전 파이프라인 검증.
II·IV·VI 3개 섹션 모두 0자 초과여야 통과.
"""

sam_row = META_INPUT_DF[
    (META_INPUT_DF["stock_code"] == "005930") &
    (META_INPUT_DF["fiscal_year"] == 2024)
].iloc[0]

print("[Sanity Check] 삼성전자 FY2024")
print(f"  stock_code : {sam_row['stock_code']}")
print(f"  corp_code  : {sam_row['corp_code']}")
print(f"  fiscal_year: {sam_row['fiscal_year']}")
print(f"  search_year: {sam_row['search_year']}")

result = collect_one_firm_year(sam_row, API_KEY, OUTPUT_DIR, verbose=True)

print(f"\n[수집 결과]")
print(f"  status    : {result['status']}")

if result["status"] == "SUCCESS":
    print(f"  rcept_no  : {result['rcept_no']}")
    print(f"  rcept_dt  : {result['rcept_dt']}")
    print(f"  report_nm : {result['report_nm']}")
    print(f"  viewer_url: {result['viewer_url']}")

    print(f"\n[섹션별 글자 수]")
    total = 0
    for sec in ["II", "IV", "VI"]:
        chars = result.get(f"section_chars_{sec}", 0)
        total += chars
        print(f"  {sec}: {chars:>8,} 자")
    print(f"  합계: {total:>8,} 자")

    print(f"\n[섹션별 텍스트 앞 200자 — ESG 표현 직접 확인]")
    xml_text, _, _ = download_document_xml(
        result["rcept_no"], API_KEY, OUTPUT_DIR, verbose=False
    )
    sections = extract_esg_sections(xml_text)
    for sec, label in [("II","사업의 내용"), ("IV","경영진단"), ("VI","이사회")]:
        txt = sections.get(sec, "")
        print(f"\n  [{sec}. {label}] {len(txt):,}자")
        print(f"  {txt[:200]}...")

    # 3개 섹션 모두 추출됐는지 검증
    assert result.get("section_chars_II", 0) > 0, "II 섹션 0자"
    assert result.get("section_chars_IV", 0) > 0, "IV 섹션 0자"
    assert result.get("section_chars_VI", 0) > 0, "VI 섹션 0자"
    print(f"\n[Sanity Check 통과] ✓ — 전체 수집 진행 가능")

else:
    print(f"  reason: {result['reason']}")
    raise AssertionError("Sanity Check 실패 — 함수 수정 필요")

[Sanity Check] 삼성전자 FY2024
  stock_code : 005930
  corp_code  : 00126380
  fiscal_year: 2024
  search_year: 2025

[수집 결과]
  status    : SUCCESS
  rcept_no  : 20250311001085
  rcept_dt  : 20250311
  report_nm : 사업보고서 (2024.12)
  viewer_url: https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20250311001085

[섹션별 글자 수]
  II:   26,457 자
  IV:   10,875 자
  VI:    5,540 자
  합계:   42,872 자

[섹션별 텍스트 앞 200자 — ESG 표현 직접 확인]

  [II. 사업의 내용] 26,457자
  당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총괄의 생산ㆍ판매법인, SDC 및 Harman 산하 종속기업 
등 228개의 종속기업으로 구성된 글로벌 전자 기업입니다.
사업별로 보면, Set 사업은 DX(Device eXperience) 부문이 TV를 비롯하여 모니터, 냉장고, 세탁기, 에어컨, 스마트폰...

  [IV. 경영진단] 10,875자
  1. 예측정보에 대한 주의사항
본 자료는 미래에 대한 '예측정보'를 포함하고 있습니다.이는 과거가 아닌 미래의 사건과 관계된 것으로 회사의 향후 예상되는 경영현황 및 재무실적을 의미하고, 표현상으로는 '예상', '전망', '계획', '기대' 등과 같은 단어를 포함합니다.'예측정보'는 그 성격상 불확실한 사건들을 언급하는데, 회사의 향후 경영현황 및 재무실적...

  [VI. 이사회] 5,540자
  가. 이사회 구성 개요
당사의 이사회는 
2024년말 현재 사내이사 3인(한종희, 노태문, 이정배)과  사외이사 6인(김한조, 김준성, 허은녕, 유명희, 신제윤, 조혜경), 총 9인의 이사로 

## **1-4. 전체 381 firm-year 수집**

### **수집 전략**

| 항목 | 결정 |
|---|---|
| Checkpoint | `collection_meta.csv` 존재 시 SUCCESS 행 skip, FAIL 행 재시도 |
| 실패 처리 | `reason` 컬럼에 사유 기록, corpus 미포함 |
| corpus 저장 | SUCCESS 행만 `outputs/corpus/{stock_code}_{fiscal_year}.json` |
| rate limit | 요청 간 `time.sleep(0.5)` 적용 |
| 저장 주기 | 매 20건마다 `collection_meta.csv` 중간 저장 |

### **FAIL 발생 시 처리 원칙**

- `보고서_미발견`: DART에 해당 연도 사업보고서 없음 (상장폐지·합병 등)
- `ZIP오류`: 원문 다운로드 실패 (DART 서버 오류 등)
- `섹션_미추출`: II·IV·VI 섹션 텍스트 추출 0건 (XML 구조 이상)

FAIL 행은 수집 완료 후 별도 진단해 재시도 가능 여부를 판단한다.

In [23]:
"""
1-4. 전체 381 firm-year 수집

Checkpoint 패턴:
- collection_meta.csv 존재 시 SUCCESS 행 skip
- FAIL 행은 재시도
- corpus JSON은 SUCCESS 행만 저장
- 실패 행은 reason 기록, corpus 미포함
"""

META_CSV = os.path.join(OUTPUT_DIR, "collection_meta.csv")

# ── Checkpoint 로드 ───────────────────────────────────────────────
if os.path.exists(META_CSV):
    done_meta = pd.read_csv(
        META_CSV, dtype={"stock_code": str, "corp_code": str}
    )
    done_meta["stock_code"]  = done_meta["stock_code"].str.zfill(6)
    done_meta["corp_code"]   = done_meta["corp_code"].fillna("").str.zfill(8).replace("00000000","")
    done_meta["fiscal_year"] = done_meta["fiscal_year"].astype(int)
    success_keys = set(zip(
        done_meta.loc[done_meta["status"]=="SUCCESS","stock_code"],
        done_meta.loc[done_meta["status"]=="SUCCESS","fiscal_year"],
    ))
    meta_records = done_meta.to_dict("records")
    print(f"[Checkpoint 로드] 기존 {len(done_meta)}행 (SUCCESS {len(success_keys)}개)")
else:
    success_keys = set()
    meta_records = []
    print(f"[Checkpoint 없음] 처음부터 수집")

# ── 전체 순회 ─────────────────────────────────────────────────────
rows    = META_INPUT_DF.to_dict("records")
n_total = len(rows)
n_skip = n_success = n_fail = 0

print(f"\n[수집 시작] 대상 {n_total}행\n")

for i, row in enumerate(rows, 1):
    key = (row["stock_code"], int(row["fiscal_year"]))

    if key in success_keys:
        n_skip += 1
        continue

    out             = collect_one_firm_year(row, API_KEY, OUTPUT_DIR, verbose=False)
    out["corp_code"] = row.get("corp_code", "")
    meta_records.append(out)

    if out["status"] == "SUCCESS":
        n_success += 1
        success_keys.add(key)
        corpus_path = os.path.join(
            CORPUS_DIR, f"{row['stock_code']}_{row['fiscal_year']}.json"
        )
        with open(corpus_path, "w", encoding="utf-8") as f:
            json.dump({
                "stock_code":       row["stock_code"],
                "fiscal_year":      int(row["fiscal_year"]),
                "rcept_no":         out["rcept_no"],
                "viewer_url":       out["viewer_url"],
                "text":             out["text"],
                "section_chars_II": out["section_chars_II"],
                "section_chars_IV": out["section_chars_IV"],
                "section_chars_VI": out["section_chars_VI"],
                "total_chars":      out["total_chars"],
            }, f, ensure_ascii=False)
    else:
        n_fail += 1

    if i % 10 == 0 or i == n_total:
        print(f"  [{i:3d}/{n_total}] {out['status']:7s} | "
              f"{row['stock_code']} FY{row['fiscal_year']} | "
              f"{out.get('reason') or out.get('total_chars',''):>10} | "
              f"성공 {n_success} 실패 {n_fail} skip {n_skip}")

    if i % 20 == 0 or i == n_total:
        pd.DataFrame(meta_records).to_csv(META_CSV, index=False, encoding="utf-8-sig")

    time.sleep(0.5)

# ── 최종 저장 ─────────────────────────────────────────────────────
META_DF = pd.DataFrame(meta_records)
META_DF["stock_code"] = META_DF["stock_code"].fillna("").str.zfill(6)
META_DF["corp_code"]  = META_DF["corp_code"].fillna("").str.zfill(8).replace("00000000","")
META_DF.to_csv(META_CSV, index=False, encoding="utf-8-sig")

# ── 수집 요약 ─────────────────────────────────────────────────────
status_counts = META_DF["status"].value_counts()
n_collected   = status_counts.get("SUCCESS", 0)
n_failed      = status_counts.get("FAIL", 0)

print(f"\n{'='*50}")
print(f"[수집 완료 요약]")
print(f"  전체 대상  : {n_total}")
print(f"  SUCCESS    : {n_collected}")
print(f"  FAIL       : {n_failed}")
print(f"  skip(기존) : {n_skip}")

print(f"\n[FAIL 행 목록]")
fail_df = META_DF[META_DF["status"]=="FAIL"][["stock_code","fiscal_year","reason"]]
if len(fail_df) > 0:
    print(fail_df.to_string(index=False))
else:
    print("  없음")

assert len(META_DF) <= 381
print(f"\n[collection_meta.csv 저장] {META_CSV}")
print(f"[META_DF] shape={META_DF.shape}")

[Checkpoint 없음] 처음부터 수집

[수집 시작] 대상 381행

  [ 10/381] SUCCESS | 000120 FY2022 |      29026 | 성공 10 실패 0 skip 0
  [ 20/381] SUCCESS | 035760 FY2023 |      38535 | 성공 20 실패 0 skip 0
  [ 30/381] SUCCESS | 000660 FY2024 |      38062 | 성공 30 실패 0 skip 0
  [ 40/381] SUCCESS | 018260 FY2022 |      29472 | 성공 40 실패 0 skip 0
  [ 50/381] SUCCESS | 051910 FY2023 |      39102 | 성공 50 실패 0 skip 0
  [ 60/381] SUCCESS | 010130 FY2024 |      42939 | 성공 60 실패 0 skip 0
  [ 70/381] SUCCESS | 105560 FY2022 |     112362 | 성공 70 실패 0 skip 0
  [ 80/381] SUCCESS | 024110 FY2023 |      36908 | 성공 80 실패 0 skip 0
  [ 90/381] SUCCESS | 002790 FY2024 |      33531 | 성공 90 실패 0 skip 0
  [100/381] SUCCESS | 011210 FY2022 |      30951 | 성공 100 실패 0 skip 0
  [110/381] SUCCESS | 042670 FY2023 |      30162 | 성공 110 실패 0 skip 0
  [120/381] SUCCESS | 000070 FY2024 |      32476 | 성공 120 실패 0 skip 0
  [130/381] SUCCESS | 000270 FY2022 |      39351 | 성공 130 실패 0 skip 0
  [140/381] SUCCESS | 000720 FY2023 |      34728 | 성공 140

**전체 381 firm-year 수집 완료**

| 항목 | 값 |
|---|---|
| 전체 대상 | 381 |
| **SUCCESS** | **381 (100%)** |
| FAIL | 0 |
| corpus 제외 행 | 없음 |

- 381/381 전수 수집 성공 — 가짜 0 처리 대상 없음
- `outputs/collection_meta.csv` 저장 (381행 × 14컬럼)
- `outputs/corpus/` 하위 firm-year별 JSON 381개 저장
- 글자 수 범위: 최소 5,693자(002600 FY2024) ~ 최대 180,540자(000880 FY2024) — 편차 약 32배. cheap-talk 통제 변수 필요성 예고

## **1-5. 수집 품질 진단**

수집한 데이터의 아래 세 가지를 추가로 확인한다.

**① 글자 수 이상치 진단**
- 너무 짧은 문서(하위 5%): 섹션 추출이 부실하거나 보고서 자체가 짧을 가능성
- 너무 긴 문서(상위 5%): 재무표 등 boilerplate가 섞였을 가능성
- 이상치 기준: 중앙값의 10% 미만 또는 10배 초과

**② 섹션별 추출 품질 확인**
- II·IV·VI 중 특정 섹션이 0자인 firm-year 존재 여부
- 0자 섹션이 있으면 해당 기업의 XML 구조가 일반적이지 않은 것

**③ 연도별·섹션별 분포 확인**
- 연도별 평균 글자 수 편차가 크면 시계열 효과 통제 필요
- 섹션별 비중이 corpus 전체에서 일관되는지 확인

In [24]:
"""
1-5. 수집 품질 진단

① 글자 수 분포 + 이상치
② 섹션별 0자 firm-year
③ 연도별·섹션별 분포
"""
import glob

# ── corpus JSON 로드 ──────────────────────────────────────────────
corpus_files = sorted(glob.glob(os.path.join(CORPUS_DIR, "*.json")))
print(f"[corpus JSON 파일 수] {len(corpus_files)}")
assert len(corpus_files) <= 381

CORPUS_DICT  = {}
char_records = []

for fpath in corpus_files:
    with open(fpath, "r", encoding="utf-8") as f:
        doc = json.load(f)
    key = (doc["stock_code"], int(doc["fiscal_year"]))
    CORPUS_DICT[key] = doc["text"]
    char_records.append({
        "stock_code":       doc["stock_code"],
        "fiscal_year":      int(doc["fiscal_year"]),
        "section_chars_II": doc.get("section_chars_II", 0),
        "section_chars_IV": doc.get("section_chars_IV", 0),
        "section_chars_VI": doc.get("section_chars_VI", 0),
        "total_chars":      doc.get("total_chars", len(doc["text"])),
    })

char_df = pd.DataFrame(char_records)
print(f"[CORPUS_DICT 로드] {len(CORPUS_DICT)} firm-year\n")

# ── ① 글자 수 분포 + 이상치 ──────────────────────────────────────
print("=" * 50)
print("[① total_chars 분포]")
desc   = char_df["total_chars"].describe()
median = char_df["total_chars"].median()
print(f"  평균   : {desc['mean']:>10,.0f} 자")
print(f"  중앙값 : {median:>10,.0f} 자")
print(f"  최소   : {desc['min']:>10,.0f} 자")
print(f"  최대   : {desc['max']:>10,.0f} 자")
print(f"  편차비 : {desc['max']/desc['min']:>10.1f} 배")

# 이상치 기준: 중앙값의 10% 미만 or 10배 초과
low_thresh  = median * 0.10
high_thresh = median * 10.0
too_short   = char_df[char_df["total_chars"] < low_thresh]
too_long    = char_df[char_df["total_chars"] > high_thresh]

print(f"\n  이상치 기준: 중앙값({median:,.0f}자)의 10% 미만 또는 10배 초과")
print(f"  너무 짧음 (<{low_thresh:,.0f}자): {len(too_short)}건")
if len(too_short) > 0:
    print(too_short[["stock_code","fiscal_year","total_chars"]].to_string(index=False))
print(f"  너무 긺  (>{high_thresh:,.0f}자): {len(too_long)}건")
if len(too_long) > 0:
    print(too_long[["stock_code","fiscal_year","total_chars"]].to_string(index=False))

# 하위·상위 5개
print(f"\n  [하위 5개]")
print(char_df.nsmallest(5,"total_chars")[
    ["stock_code","fiscal_year","total_chars"]].to_string(index=False))
print(f"\n  [상위 5개]")
print(char_df.nlargest(5,"total_chars")[
    ["stock_code","fiscal_year","total_chars"]].to_string(index=False))

# ── ② 섹션별 0자 firm-year ───────────────────────────────────────
print(f"\n{'='*50}")
print("[② 섹션별 0자 firm-year]")
for sec in ["II", "IV", "VI"]:
    col  = f"section_chars_{sec}"
    zero = char_df[char_df[col] == 0]
    print(f"  {sec} 섹션 0자: {len(zero)}건", end="")
    if len(zero) > 0:
        codes = zero[["stock_code","fiscal_year"]].values.tolist()
        print(f" → {codes[:5]}")
    else:
        print()

# ── ③ 연도별·섹션별 분포 ─────────────────────────────────────────
print(f"\n{'='*50}")
print("[③ 연도별 평균 글자 수]")
yr_stats = char_df.groupby("fiscal_year")["total_chars"].agg(["mean","median","min","max"])
print(yr_stats.round(0).to_string())

print(f"\n[섹션별 평균 글자 수 및 비중]")
total_mean = char_df["total_chars"].mean()
for sec in ["II","IV","VI"]:
    col  = f"section_chars_{sec}"
    mean = char_df[col].mean()
    zero = (char_df[col] == 0).sum()
    print(f"  {sec}: {mean:>8,.0f}자 ({mean/total_mean*100:.1f}%)  |  0자 firm-year: {zero}건")

print(f"\n[CORPUS_DICT] {len(CORPUS_DICT)} firm-year 메모리 적재 완료")

[corpus JSON 파일 수] 381
[CORPUS_DICT 로드] 381 firm-year

[① total_chars 분포]
  평균   :     34,773 자
  중앙값 :     25,872 자
  최소   :      4,513 자
  최대   :    189,606 자
  편차비 :       42.0 배

  이상치 기준: 중앙값(25,872자)의 10% 미만 또는 10배 초과
  너무 짧음 (<2,587자): 0건
  너무 긺  (>258,720자): 0건

  [하위 5개]
stock_code  fiscal_year  total_chars
    000650         2024         4513
    000650         2022         4688
    000650         2023         4690
    002600         2024         5693
    002600         2023         5750

  [상위 5개]
stock_code  fiscal_year  total_chars
    005490         2022       189606
    005490         2023       184505
    000880         2024       180540
    005490         2024       173772
    000880         2023       167458

[② 섹션별 0자 firm-year]
  II 섹션 0자: 0건
  IV 섹션 0자: 0건
  VI 섹션 0자: 0건

[③ 연도별 평균 글자 수]
                mean   median   min     max
fiscal_year                                
2022         34165.0  26322.0  4688  189606
2023         35423.0  25851.0  4690  184505
2024

**수집 품질 진단 완료**

**① 글자 수 분포**

| 항목 | 값 |
|---|---|
| 평균 | 34,773자 |
| 중앙값 | 25,872자 |
| 최소 | 4,513자 (000650 FY2024) |
| 최대 | 189,606자 (005490 FY2022) |
| 편차비 | 42.0배 |

- 이상치 기준(중앙값 10% 미만 · 10배 초과) 해당 firm-year 없음
- 하위 5개(000650·002600)는 4,500~5,800자 수준 — 이상치 기준은 통과하나 상대적으로 짧음. 해당 기업의 사업보고서가 원래 간결한 구조일 가능성이 높음 (보고서 직접 확인 권장)
- 상위 5개(005490·000880)는 170,000~190,000자 — 대기업 계열사로 보고서 자체가 방대한 구조. TABLE 제거 후에도 II 섹션 비중이 높아 ESG 외 사업 서술이 포함될 수 있음
- **편차 42배는 cheap-talk 통제 변수(`log_total_chars`) 필수 투입 근거**

**② 섹션별 0자 firm-year**

- II·IV·VI 전 섹션에서 0자 firm-year 없음 — 섹션 추출 함수 정상 작동 확인

**③ 연도별 분포**

| 연도 | 평균 | 중앙값 |
|---|---|---|
| FY2022 | 34,165자 | 26,322자 |
| FY2023 | 35,423자 | 25,851자 |
| FY2024 | 34,731자 | 25,872자 |

- 연도별 평균 편차 최대 1,258자(3.7%) — 연도 간 systematic bias 낮음
- 섹션별 비중: II 61.4% · IV 30.1% · VI 8.5% — 전 연도 일관

381 firm-year 전수 수집 완료, 이상치·섹션 누락 없음. `CORPUS_DICT` 메모리 적재 완료.

---

# **2. 전처리 — 형태소 분석기 선택 · Seed 보호 · 불용어**

## **2-1. 형태소 분석기 선택**

한국어 TF-IDF 분석의 품질은 형태소 분석기 선택에 크게 의존한다. ESG seed 30개 중 상당수가 `재생에너지`, `감사위원회`, `탄소중립`처럼 복합명사이므로 분석기가 이를 단일 토큰으로 보존하는가를 핵심 선택 기준으로 삼는다.

분석기를 결정하기 전 Kiwi와 Okt를 동일한 corpus 샘플에서 직접 비교한다.

### **비교 기준**

| 기준 | 내용 |
|---|---|
| seed 보존율 | seed 30개가 단일 토큰으로 추출되는 비율 |
| 복합명사 처리 | `재생에너지` → `재생에너지`(보존) vs `재생` + `에너지`(분리) |
| 속도 | corpus 샘플 30개 토큰화 소요 시간 |
| 사용자 사전 | seed 등록 후 보존율 변화 |

### **Decision Box ⑦ — 형태소 분석기**

| 옵션 | seed 보존 | 복합명사 처리 | 속도 | 사용자 사전 |
|---|---|---|---|---|
| Kiwi | 실험으로 확인 | 실험으로 확인 | 실험으로 확인 | `add_user_word` 지원 |
| Okt | 실험으로 확인 | 실험으로 확인 | 실험으로 확인 | 미지원 |

Pecab·Komoran·Kkma는 속도와 사용 편의성 문제로 비교 대상에서 제외한다.  
Kiwi와 Okt를 corpus 샘플 30개로 직접 비교한 뒤 채택을 결정한다.

In [25]:
"""
2-1. 형태소 분석기 비교 — Kiwi vs Okt

corpus 샘플 30개에서:
① seed 보존율 (사용자 사전 등록 전/후)
② 복합명사 분리 사례
③ 토큰화 속도
"""

# ── 설치 확인 ─────────────────────────────────────────────────────
try:
    from kiwipiepy import Kiwi
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "kiwipiepy"], check=True)
    from kiwipiepy import Kiwi

try:
    from konlpy.tag import Okt
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "konlpy"], check=True)
    from konlpy.tag import Okt

# ── Seed 30개 ─────────────────────────────────────────────────────
SEED_DICT = {
    "E": ["탄소","온실가스","탄소중립","넷제로","재생에너지",
          "에너지","전력","폐기물","재활용","폐수"],
    "S": ["안전","산업재해","중대재해","임직원","노동",
          "인권","교육훈련","협력사","공급망","지역사회"],
    "G": ["이사회","사외이사","감사위원회","독립성","윤리",
          "준법","컴플라이언스","부패방지","주주","의결권"],
}
ALL_SEEDS = [t for terms in SEED_DICT.values() for t in terms]

# ── corpus 샘플 30개 (글자 수 분위수 기준) ────────────────────────
np.random.seed(SEED)
sample_keys = sorted(CORPUS_DICT.keys())
total_chars  = [len(CORPUS_DICT[k]) for k in sample_keys]
indices      = np.argsort(total_chars)
step         = len(indices) // 30
sample_idx   = [indices[i * step] for i in range(30)]
sample_texts = [CORPUS_DICT[sample_keys[i]] for i in sample_idx]
print(f"[샘플] {len(sample_texts)}개 (글자 수 분위수 기준)")
print(f"  최소: {min(len(t) for t in sample_texts):,}자  "
      f"최대: {max(len(t) for t in sample_texts):,}자\n")

# ── Kiwi 초기화 (사용자 사전 없는 버전 먼저) ──────────────────────
kiwi_base = Kiwi()
kiwi_user = Kiwi()
for term in ALL_SEEDS:
    kiwi_user.add_user_word(term, "NNP", score=50.0)

def extract_nouns_kiwi(text, kiwi_inst, min_len=2):
    return [t.form for t in kiwi_inst.tokenize(text)
            if t.tag in ("NNG","NNP") and len(t.form) >= min_len]

# ── Okt 초기화 ────────────────────────────────────────────────────
okt = Okt()

def extract_nouns_okt(text, min_len=2):
    return [t for t in okt.nouns(text) if len(t) >= min_len]

# ── ① seed 보존율 비교 ────────────────────────────────────────────
print("=" * 55)
print("[① seed 보존율 — 샘플 30개 합산 토큰에서 seed 등장 여부]")

# 샘플 전체 합산 텍스트로 비교
combined_sample = "\n".join(sample_texts)

kiwi_base_tokens = set(extract_nouns_kiwi(combined_sample, kiwi_base))
kiwi_user_tokens = set(extract_nouns_kiwi(combined_sample, kiwi_user))
okt_tokens       = set(extract_nouns_okt(combined_sample))

print(f"\n  {'seed':<12} {'Kiwi(기본)':<12} {'Kiwi(사전)':<12} {'Okt':<12}")
print(f"  {'-'*48}")
for seed in ALL_SEEDS:
    kb = "✓" if seed in kiwi_base_tokens else "✗"
    ku = "✓" if seed in kiwi_user_tokens else "✗"
    ok = "✓" if seed in okt_tokens       else "✗"
    flag = " ←" if kb != ku else ""
    print(f"  {seed:<12} {kb:<12} {ku:<12} {ok:<12}{flag}")

n_kb = sum(1 for s in ALL_SEEDS if s in kiwi_base_tokens)
n_ku = sum(1 for s in ALL_SEEDS if s in kiwi_user_tokens)
n_ok = sum(1 for s in ALL_SEEDS if s in okt_tokens)
print(f"\n  보존 합계: Kiwi(기본) {n_kb}/30  "
      f"Kiwi(사전) {n_ku}/30  Okt {n_ok}/30")

# ── ② 복합명사 분리 사례 ──────────────────────────────────────────
print(f"\n{'='*55}")
print("[② 복합명사 분리 사례 비교]")
test_cases = [
    ("재생에너지 사용 비중을 확대하였다.", "재생에너지"),
    ("감사위원회의 독립성을 강화하였다.", "감사위원회"),
    ("탄소중립 목표를 추진하고 있다.", "탄소중립"),
    ("공급망 전반에 걸쳐 ESG 실사를 실시하였다.", "공급망"),
    ("중대재해 발생률을 관리하였다.", "중대재해"),
    ("컴플라이언스 체계를 강화하였다.", "컴플라이언스"),
]
print(f"\n  {'seed':<12} {'문장 토큰 (Kiwi기본)':<30} {'문장 토큰 (Kiwi사전)':<30} {'문장 토큰 (Okt)'}")
print(f"  {'-'*100}")
for sent, seed in test_cases:
    kb = extract_nouns_kiwi(sent, kiwi_base)
    ku = extract_nouns_kiwi(sent, kiwi_user)
    ok = extract_nouns_okt(sent)
    print(f"  {seed:<12} {str(kb):<30} {str(ku):<30} {str(ok)}")

# ── ③ 속도 비교 ───────────────────────────────────────────────────
print(f"\n{'='*55}")
print("[③ 속도 비교 — 샘플 30개]")

t0 = time.time()
for t in sample_texts:
    extract_nouns_kiwi(t, kiwi_user)
t_kiwi = time.time() - t0

t0 = time.time()
for t in sample_texts:
    extract_nouns_okt(t)
t_okt = time.time() - t0

print(f"  Kiwi(사전): {t_kiwi:.2f}초")
print(f"  Okt       : {t_okt:.2f}초")
print(f"  속도 비율 : Okt가 Kiwi 대비 {t_okt/t_kiwi:.1f}배 {'느림' if t_okt > t_kiwi else '빠름'}")

[샘플] 30개 (글자 수 분위수 기준)
  최소: 4,513자  최대: 74,324자

[① seed 보존율 — 샘플 30개 합산 토큰에서 seed 등장 여부]

  seed         Kiwi(기본)     Kiwi(사전)     Okt         
  ------------------------------------------------
  탄소           ✓            ✓            ✓           
  온실가스         ✓            ✓            ✓           
  탄소중립         ✗            ✓            ✗            ←
  넷제로          ✗            ✗            ✗           
  재생에너지        ✗            ✓            ✓            ←
  에너지          ✓            ✓            ✓           
  전력           ✓            ✓            ✓           
  폐기물          ✓            ✓            ✓           
  재활용          ✓            ✓            ✓           
  폐수           ✓            ✓            ✓           
  안전           ✓            ✓            ✓           
  산업재해         ✗            ✓            ✗            ←
  중대재해         ✗            ✓            ✗            ←
  임직원          ✓            ✓            ✗           
  노동           ✓            ✓          

**Kiwi vs Okt 비교 결과**

**① seed 보존율**

| 분석기 | 보존 수 | 보존율 |
|---|---|---|
| Kiwi (기본, 사용자 사전 없음) | 18/30 | 60% |
| **Kiwi (사용자 사전 등록)** | **28/30** | **93%** |
| Okt | 20/30 | 67% |

- 미보존 seed (Kiwi 사전 등록 후에도): `넷제로`(2건), `부패방지`(2건) — 샘플 30개에 해당 단어가 아예 등장하지 않을 가능성 높음, 전체 corpus에서 재확인 필요
- Okt 미보존 패턴: `임직원`, `협력사`, `산업재해`, `중대재해`, `교육훈련`, `지역사회`, `감사위원회` — S·G 차원 핵심 어휘 다수 분리

**② 복합명사 분리 사례**

| seed | Kiwi 기본 | Kiwi 사전 | Okt |
|---|---|---|---|
| 재생에너지 | 재생 + 에너지 ✗ | **재생에너지** ✓ | 재생에너지 ✓ |
| 감사위원회 | 감사 + 위원회 ✗ | **감사위원회** ✓ | 감사 + 위원회 ✗ |
| 탄소중립 | 탄소 + 중립 ✗ | **탄소중립** ✓ | 탄소 + 중립 ✗ |
| 중대재해 | 중대 + 재해 ✗ | **중대재해** ✓ | 중대 + 재해 ✗ |
| 컴플라이언스 | 플라이 + 언스 ✗ | **컴플라이언스** ✓ | 컴플라이언스 ✓ |

- **Kiwi 기본**은 외래어(`컴플라이언스`)를 형태소 단위로 분리 — 사용 불가 수준
- **Okt**는 `감사위원회`, `탄소중립`, `중대재해` 분리 — G·S 핵심 seed 신호 손실
- **Kiwi + 사용자 사전**은 6개 중 6개 보존

**③ 속도**

| 분석기 | 샘플 30개 소요 시간 |
|---|---|
| Kiwi (사용자 사전) | 48.64초 |
| Okt | 32.70초 |

- Okt가 약 1.5배 빠르나 전체 381건 기준 예상 추가 소요는 약 3~4분 — 허용 범위

### **Decision Box ⑦ 채택 결정 — 형태소 분석기**

**채택: Kiwi + 사용자 사전 등록 (score=50.0)**

| 기준 | Kiwi (사전) | Okt | 판정 |
|---|---|---|---|
| seed 보존율 | **28/30 (93%)** | 20/30 (67%) | Kiwi 우세 |
| 복합명사 처리 | **6/6 보존** | 3/6 보존 | Kiwi 우세 |
| 속도 | 48.64초 | 32.70초 | Okt 우세 |
| 사용자 사전 | **지원** | 미지원 | Kiwi 우세 |

**채택 근거**

- seed 보존율이 TF-IDF 신호 품질의 핵심 — 60% vs 93% 차이는 G·S 차원 feature에 직접 영향
- Okt는 `감사위원회`(G 핵심)·`중대재해`·`산업재해`(S 핵심)를 분리 → G·S seed score 심각하게 저하
- Kiwi 기본도 복합명사를 분리하지만 `add_user_word(score=50.0)`으로 seed 30개를 등록하면 단일 토큰 보존 강제 가능
- 속도 열위(약 16초 차이)는 전체 381건에서 약 3~4분 추가 — 분석 정확도 대비 감수 가능한 수준

**비채택 근거**

- Okt: 사용자 사전 미지원 — 복합명사 분리 문제를 구조적으로 해결 불가
- Komoran: 사용자 사전 파일 기반으로 등록 절차 복잡, 속도 느림
- Kkma: 속도가 매우 느려 381 firm-year 전체 처리에 비현실적

**미보존 seed 2개 (`넷제로`, `부패방지`) 처리 방침**

- 샘플 30개에 해당 단어가 등장하지 않아 보존 여부 미확인
- 전체 corpus 토큰화 후 seed 빈도 진단에서 재확인
- 빈도가 매우 낮으면 expanded dictionary에서 유사어로 보강

## **2-2. Kiwi 사용자 사전 등록 + 불용어 구축**

### **Decision Box ⑧ — 토큰화 단위**

- 명사만 (NNG/NNP), min_len=2
- ESG seed 30개가 모두 명사 — 다른 품사 추가 시 TF-IDF 가중치만 분산
- 1글자 명사 제외(`min_len=2`) — "이", "수", "등" 단음절 노이즈 제거

### **Decision Box ⑨ — 불용어 정책**

**3종 통합 + seed 강제 보호**

| 종류 | 구성 방식 | 규모 | 목적 |
|---|---|---|---|
| 기본 불용어 | 인라인 정의 | 78개 | 조사·부사·일반 경영어 제거 |
| 회사명 토큰 | company_master 회사명 Kiwi 분해 | 132개 | 기업 고유명사 노이즈 제거 |
| 보일러플레이트 | 사업보고서 공통 상투어 | 30개 | 재무·공시 형식어 제거 |

**seed 강제 보호 안전장치**

- 회사명 분해 시 `에너지`(한화에너지·SK에너지 등)가 불용어에 혼입 가능
- 최종 불용어에서 seed 30개 강제 제외 후 `assert not (STOPWORDS & set(ALL_SEEDS))` 검증
- 이 안전장치 없으면 E 차원 seed 신호 손실

In [26]:
"""
2-2. Kiwi 사용자 사전 등록 + 불용어 구축

Decision Box ⑨ 채택: Kiwi + 사용자 사전 (score=50.0)
- seed 30개 등록 후 보존 검증
- 불용어 3종 통합 + seed 강제 보호
"""

# ── Kiwi 초기화 + seed 등록 ───────────────────────────────────────
kiwi = Kiwi()
for term in ALL_SEEDS:
    kiwi.add_user_word(term, "NNP", score=50.0)
print(f"[Kiwi 사용자 사전 등록] {len(ALL_SEEDS)}개 seed 등록 완료")

# ── 명사 추출 함수 (이후 전 단계 재사용) ──────────────────────────
def extract_nouns_kiwi(text, kiwi_inst=None, min_len=2):
    if kiwi_inst is None:
        kiwi_inst = kiwi
    return [t.form for t in kiwi_inst.tokenize(text)
            if t.tag in ("NNG","NNP") and len(t.form) >= min_len]

# ── seed 보존 검증 (까다로운 컨텍스트 8개) ───────────────────────
print(f"\n[seed 보존 검증 — 조사·접미사 포함 문장]")
test_cases = [
    ("재생에너지",   "당사는 재생에너지 사용 비중을 확대하고 있습니다."),
    ("감사위원회",   "감사위원회의 독립성과 전문성을 강화하였다."),
    ("공급망",      "공급망 전체에 대한 ESG 실사를 실시하였다."),
    ("온실가스",    "온실가스 배출량을 전년 대비 8% 감축하였다."),
    ("탄소중립",    "2030년 탄소중립 목표를 단계적으로 추진한다."),
    ("산업재해",    "산업재해 발생률을 0.05% 미만으로 관리하였다."),
    ("협력사",      "1차 협력사에 대한 안전보건 교육을 강화하였다."),
    ("이사회",      "이사회 산하 ESG위원회는 사외이사로 구성된다."),
]
n_pass = 0
for seed, sent in test_cases:
    tokens    = extract_nouns_kiwi(sent)
    preserved = seed in tokens
    if preserved:
        n_pass += 1
    print(f"  {'✓' if preserved else '✗'} [{seed:<8}] → {tokens}")
print(f"\n  결과: {n_pass}/{len(test_cases)} 통과")
assert n_pass == len(test_cases), "일부 seed 보존 실패 — 사전 등록 점검 필요"
print(f"  → 모든 컨텍스트에서 seed 단일 토큰 보존 ✓")

# ── 불용어 구축 (3종 통합 + seed 보호) ───────────────────────────
print(f"\n{'='*55}")
print("[불용어 구축 — 3종 통합]")

# 1) 기본 불용어
default_sw = {
    "및","또는","그리고","그러나","하지만","따라서","관련","통해","위해",
    "대한","대해","있는","있다","없다","이","그","저","것","수","등","들",
    "사용","수행","운영","관리","실시","진행","지원","제공","수립","추진",
    "구축","확보","강화","지속","발전","성장","노력","활동","사항","내용",
    "방안","결과","현황","상황","정도","수준","범위","분야","체계","확대",
    "축소","변경","개선","보완","검토","예정","기간","단계","과정","절차",
    "당사","본사","회사","기업","업체","법인","사업","영업","경영","계획",
    "전략","정책","시장","산업","부문","지역","국가",
}
print(f"  기본 불용어      : {len(default_sw)}개")

# 2) 회사명 토큰
company_tokens = set()
for name in META_INPUT_DF["company_name"].unique():
    company_tokens.update(extract_nouns_kiwi(name))
# seed와 교집합 확인
seeds_in_company = company_tokens & set(ALL_SEEDS)
if seeds_in_company:
    print(f"  ⚠ 회사명에 seed 포함: {sorted(seeds_in_company)} → 자동 제외 예정")
print(f"  회사명 토큰      : {len(company_tokens)}개")

# 3) 보일러플레이트
boilerplate = {
    "백만원","천원","달러","원화","외화","환율",
    "결산","회계","재무","손익","자산","부채","자본",
    "주식","주가","배당","이익","매출","비용","수익",
    "보고서","공시","신고","제출","기재","정정",
    "별첨","참조","주석","단위",
}
print(f"  보일러플레이트   : {len(boilerplate)}개")

# 통합
STOPWORDS_RAW = default_sw | company_tokens | boilerplate

# seed 강제 보호
seeds_in_sw = STOPWORDS_RAW & set(ALL_SEEDS)
if seeds_in_sw:
    print(f"\n  ⚠ 불용어에 seed 포함 → 강제 제외: {sorted(seeds_in_sw)}")
STOPWORDS = STOPWORDS_RAW - set(ALL_SEEDS)

# 검증
assert not (STOPWORDS & set(ALL_SEEDS)), "seed가 불용어에 잔존"
print(f"\n  통합 불용어 (보호 전) : {len(STOPWORDS_RAW)}개")
print(f"  seed 제외 후 최종    : {len(STOPWORDS)}개")
print(f"  → 모든 seed가 불용어에서 제외됨 ✓")

[Kiwi 사용자 사전 등록] 30개 seed 등록 완료

[seed 보존 검증 — 조사·접미사 포함 문장]
  ✓ [재생에너지   ] → ['당사', '재생에너지', '사용', '비중', '확대']
  ✓ [감사위원회   ] → ['감사위원회', '독립성', '전문', '강화']
  ✓ [공급망     ] → ['공급망', '전체', '실사', '실시']
  ✓ [온실가스    ] → ['온실가스', '배출량', '전년', '대비', '감축']
  ✓ [탄소중립    ] → ['탄소중립', '목표', '단계', '추진']
  ✓ [산업재해    ] → ['산업재해', '발생', '미만', '관리']
  ✓ [협력사     ] → ['협력사', '안전', '보건', '교육', '강화']
  ✓ [이사회     ] → ['이사회', '산하', '위원회', '사외이사', '구성']

  결과: 8/8 통과
  → 모든 컨텍스트에서 seed 단일 토큰 보존 ✓

[불용어 구축 — 3종 통합]
  기본 불용어      : 78개
  ⚠ 회사명에 seed 포함: ['에너지'] → 자동 제외 예정
  회사명 토큰      : 132개
  보일러플레이트   : 30개

  ⚠ 불용어에 seed 포함 → 강제 제외: ['에너지']

  통합 불용어 (보호 전) : 238개
  seed 제외 후 최종    : 237개
  → 모든 seed가 불용어에서 제외됨 ✓


**Kiwi 사용자 사전 등록 + 불용어 구축 완료**

**seed 보존 검증**: 8/8 통과 — 조사·접미사 포함 실제 문장에서 모든 seed 단일 토큰 보존 확인

**불용어 구축 결과**

| 종류 | 개수 |
|---|---|
| 기본 불용어 | 78개 |
| 회사명 토큰 | 132개 |
| 보일러플레이트 | 30개 |
| 통합 (중복 제거 전) | 238개 |
| **최종 (seed 제외 후)** | **237개** |

**seed 보호 안전장치 작동 확인**

- `에너지`가 회사명 토큰(한화에너지, SK에너지 등)에서 추출되어 불용어에 혼입
- seed 강제 제외 로직으로 자동 제거 → E 차원 seed 신호 손실 방지
- 이 안전장치가 없었으면 `에너지` TF-IDF 신호가 0이 되어 E feature 전체에 영향

## **2-3. 전체 corpus 토큰화**

Kiwi(사용자 사전) + 불용어 237개를 적용해 381 firm-year 전체를 토큰화한다.

- Checkpoint: `outputs/tokens_kiwi.json` 존재 시 즉시 로드, 없으면 토큰화 후 저장
- 토큰화 완료 후 아래 4가지를 진단해 3장 Feature 생성의 입력 품질을 확인한다

| 진단 항목 | 목적 |
|---|---|
| firm-year별 토큰 수 분포 | 글자 수와 상관 확인, cheap-talk 통제 변수 예고 |
| seed 30개 빈도 진단 | 희소 seed 발견 → expanded dictionary 필요성 정량 근거 |
| 고빈도 토큰 Top 30 | 추가 불용어 후보 발견 |
| 차원별(E·S·G) 빈도 불균형 | E·S·G raw 합계 직접 비교 금지 근거 |

In [ ]:
"""
2-3. 전체 corpus 토큰화 + 품질 진단

Checkpoint: outputs/tokens_kiwi.json 존재 시 로드
진단 4종:
① firm-year별 토큰 수 분포
② seed 30개 빈도 (희소 seed 발견)
③ 고빈도 토큰 Top 30
④ 차원별 빈도 불균형
"""

TOKENS_PATH = os.path.join(OUTPUT_DIR, "tokens_kiwi.json")

# ── Checkpoint ────────────────────────────────────────────────────
if os.path.exists(TOKENS_PATH):
    print(f"[Checkpoint] tokens_kiwi.json 존재 — 로드")
    with open(TOKENS_PATH, "r", encoding="utf-8") as f:
        tokens_raw = json.load(f)
    TOKEN_DICT = {
        (k.split("_")[0], int(k.split("_")[1])): v
        for k, v in tokens_raw.items()
    }
    print(f"  로드 완료: {len(TOKEN_DICT)} firm-year")
else:
    print(f"[토큰화 시작] {len(CORPUS_DICT)} firm-year")
    print(f"  예상 시간: 약 10~20분 (Kiwi C++, 381건)")
    TOKEN_DICT = {}
    t_start    = time.time()

    for i, (key, text) in enumerate(CORPUS_DICT.items(), 1):
        nouns    = extract_nouns_kiwi(text, kiwi)
        filtered = [t for t in nouns if t not in STOPWORDS]
        TOKEN_DICT[key] = filtered

        if i % 50 == 0 or i == len(CORPUS_DICT):
            elapsed   = time.time() - t_start
            remaining = (len(CORPUS_DICT) - i) / (i / elapsed) if elapsed > 0 else 0
            print(f"  [{i:3d}/{len(CORPUS_DICT)}] {key} → {len(filtered):,} tokens "
                  f"| 경과 {elapsed:.0f}s 잔여 {remaining:.0f}s")

    # 저장
    tokens_for_save = {f"{k[0]}_{k[1]}": v for k, v in TOKEN_DICT.items()}
    with open(TOKENS_PATH, "w", encoding="utf-8") as f:
        json.dump(tokens_for_save, f, ensure_ascii=False)
    print(f"\n  저장: {TOKENS_PATH}")
    print(f"  소요: {(time.time()-t_start)/60:.1f}분")

# ── 검증 ──────────────────────────────────────────────────────────
n_tokens = len(TOKEN_DICT)
n_corpus = len(CORPUS_DICT)
assert n_tokens == n_corpus, f"TOKEN_DICT({n_tokens}) ≠ CORPUS_DICT({n_corpus})"
assert n_tokens <= 381
print(f"\n[TOKEN_DICT] {n_tokens} firm-year ✓")

# ── ① firm-year별 토큰 수 분포 ────────────────────────────────────
print(f"\n{'='*55}")
print("[① firm-year별 토큰 수 분포]")
token_counts = pd.DataFrame([
    {"stock_code": k[0], "fiscal_year": k[1],
     "n_tokens": len(v), "n_chars": len(CORPUS_DICT[k])}
    for k, v in TOKEN_DICT.items()
])
desc = token_counts["n_tokens"].describe()
print(f"  평균   : {desc['mean']:>8,.0f} tokens")
print(f"  중앙값 : {token_counts['n_tokens'].median():>8,.0f}")
print(f"  최소   : {desc['min']:>8,.0f}")
print(f"  최대   : {desc['max']:>8,.0f}")
print(f"  편차비 : {desc['max']/desc['min']:>8.1f} 배")
corr = token_counts["n_chars"].corr(token_counts["n_tokens"])
print(f"  글자수-토큰수 상관: {corr:.3f} (정합성 확인, ≥0.95 정상)")
assert corr > 0.90, "상관 낮음 — 토큰화 이상 가능성"

# ── ② seed 빈도 진단 ──────────────────────────────────────────────
print(f"\n{'='*55}")
print("[② seed 30개 빈도 진단]")
all_tokens     = [t for tokens in TOKEN_DICT.values() for t in tokens]
token_counter  = Counter(all_tokens)

seed_freq = []
for dim, seeds in SEED_DICT.items():
    for s in seeds:
        seed_freq.append({
            "dim":    dim,
            "seed":   s,
            "freq":   token_counter.get(s, 0),
            "n_docs": sum(1 for tokens in TOKEN_DICT.values() if s in tokens),
        })
seed_freq_df = pd.DataFrame(seed_freq).sort_values("freq", ascending=False)
print(seed_freq_df.to_string(index=False))

sparse = seed_freq_df[
    (seed_freq_df["freq"] < 20) | (seed_freq_df["n_docs"] < int(n_tokens * 0.10))
]
print(f"\n  희소 seed (빈도<20 또는 등장문서<10%): {len(sparse)}개")
if len(sparse) > 0:
    print(sparse.to_string(index=False))
print(f"  → expanded dictionary 필요성의 정량 근거")

# ── ③ 고빈도 토큰 Top 30 ──────────────────────────────────────────
print(f"\n{'='*55}")
print("[③ 고빈도 토큰 Top 30 — 추가 불용어 후보]")
for token, cnt in token_counter.most_common(30):
    is_seed = " ★seed" if token in ALL_SEEDS else ""
    print(f"  {token:<15} {cnt:>8,}{is_seed}")

# ── ④ 차원별 빈도 불균형 ──────────────────────────────────────────
print(f"\n{'='*55}")
print("[④ 차원별 seed 빈도 불균형]")
dim_freq = {}
for dim, seeds in SEED_DICT.items():
    dim_freq[dim] = sum(token_counter.get(s, 0) for s in seeds)
total_freq = sum(dim_freq.values())
for dim in ["E","S","G"]:
    pct = dim_freq[dim] / total_freq * 100
    print(f"  {dim}: {dim_freq[dim]:>8,} ({pct:.1f}%)")
imbalance = max(dim_freq.values()) / max(min(dim_freq.values()), 1)
print(f"\n  최대/최소 비율: {imbalance:.1f}배")
print(f"  → E·S·G raw 합계 직접 비교 금지, 차원별 별도 회귀 권장")

[토큰화 시작] 381 firm-year
  예상 시간: 약 10~20분 (Kiwi C++, 381건)
  [ 50/381] ('000320', 2023) → 3,879 tokens | 경과 74s 잔여 489s
  [100/381] ('000720', 2022) → 4,858 tokens | 경과 110s 잔여 309s
  [150/381] ('001120', 2024) → 6,540 tokens | 경과 216s 잔여 332s
  [200/381] ('001450', 2023) → 4,549 tokens | 경과 253s 잔여 229s
  [250/381] ('002790', 2022) → 7,630 tokens | 경과 300s 잔여 157s
  [300/381] ('006400', 2024) → 5,339 tokens | 경과 386s 잔여 104s
  [350/381] ('055550', 2023) → 16,850 tokens | 경과 471s 잔여 42s
  [381/381] ('316140', 2024) → 15,713 tokens | 경과 639s 잔여 0s

  저장: outputs\tokens_kiwi.json
  소요: 10.7분

[TOKEN_DICT] 381 firm-year ✓

[① firm-year별 토큰 수 분포]
  평균   :    4,987 tokens
  중앙값 :    3,761
  최소   :      587
  최대   :   26,804
  편차비 :     45.7 배
  글자수-토큰수 상관: 0.998 (정합성 확인, ≥0.95 정상)

[② seed 30개 빈도 진단]
dim   seed  freq  n_docs
  G    이사회  7928     381
  G   사외이사  7459     381
  G     주주  7332     381
  G  감사위원회  4249     362
  E    에너지  2851     238
  S     안전  1868     292
  G     준법  1676   

**전처리 완료 — TOKEN_DICT 381 firm-year**

**① 토큰 수 분포**

| 항목 | 값 |
|---|---|
| 평균 | 4,987 tokens |
| 중앙값 | 3,761 tokens |
| 최소 | 587 tokens |
| 최대 | 26,804 tokens |
| 편차비 | **45.7배** |
| 글자수-토큰수 상관 | **0.998** (토큰화 정합성 확인) |

- 편차 45.7배 — 1장의 글자 수 편차(42배)와 일관, 토큰화 이상 없음
- **cheap-talk 통제 변수(`log_n_tokens`) 필수 투입 근거 정량 확보**

**② seed 30개 빈도 진단**

- G seed 핵심 3개(`이사회` 7,928 · `사외이사` 7,459 · `주주` 7,332)가 381/381 전 문서에 등장 — 법정 의무 공시 어휘로 변별력 없음. TF-IDF의 `max_df` 설정에서 자동 하향 처리되며 이는 G cheap-talk 분석의 핵심 근거
- 희소 seed 7개 (빈도<20 또는 등장 문서 < 10%)

| 차원 | seed | 빈도 | 등장 문서 수 | 해석 |
|---|---|---|---|---|
| S | 인권 | 99 | 35 (9%) | S 핵심 어휘인데 희소 |
| S | 중대재해 | 74 | 27 (7%) | 법적 용어, 일반 보고서에서 드묾 |
| G | 컴플라이언스 | 50 | 32 (8%) | 영어 차용어, "준법"으로 대체 표현 |
| S | 산업재해 | 48 | 20 (5%) | "안전사고" 등 대체 표현 가능성 |
| S | 교육훈련 | 17 | 17 (4%) | 빈도 매우 낮음 |
| E | 넷제로 | 9 | 7 (2%) | 신조어, 표준 표기 미확립 |
| G | 부패방지 | 8 | 5 (1%) | "윤리경영"으로 대체 표현 |

- seed-only 분석으로는 특히 S 신호가 매우 약함 → FastText 확장으로 유사어 보강 필수

**③ 고빈도 토큰 Top 30**

- `위험`(22,639) · `기준`(18,361) · `연결`(17,431) 등 ESG 무관 일반 경영어가 최상위
- TF-IDF의 IDF 가중치가 자동으로 이들을 하향 처리 — 추가 불용어 등록 불필요
- seed 3개(`이사회` · `사외이사` · `주주`)가 Top 30 내 등장 — G cheap-talk 직접 증거

**④ 차원별 빈도 불균형**

| 차원 | seed 빈도 합계 | 비중 |
|---|---|---|
| E | 8,859 | 20.1% |
| S | 3,518 | 8.0% |
| G | 31,623 | 71.9% |

- 최대/최소 비율 9.0배 — G가 압도적인 이유는 `이사회`·`사외이사`·`주주`의 법정 의무 공시
- E·S·G raw 합계 직접 비교 금지. 차원 내 firm-year 상대 비교 + 차원별 별도 회귀 권장

**결론**
Kiwi(사용자 사전) 채택, 불용어 237개, TOKEN_DICT 381 firm-year 생성 완료.  
희소 seed 7개 발견 → FastText 확장 필요성 정량 근거 확보

---

# **3. Feature 생성 — TF-IDF · FastText · Cosine**

## **3-1. TF-IDF 파라미터 설계**

TOKEN_DICT의 토큰화 결과를 TF-IDF 행렬로 변환한다.  
파라미터 선택이 seed vocab 포함 여부와 feature 품질에 직접 영향을 미치므로 2단계의 진단 결과를 근거로 각 파라미터를 결정한다.

### **Decision Box ⑩ — TF-IDF 파라미터**

**`min_df` — 최소 등장 문서 수**

| 옵션 | 값 | 영향 |
|---|---|---|
| 1 | 모든 단어 포함 | 1회 등장 오탈자·노이즈 포함 |
| 5 | 5개 미만 문서 제외 | 희소 노이즈 컷, 희소 seed(넷제로 7건·부패방지 5건) 일부 제외 |
| 10 | 10개 미만 문서 제외 | 희소 seed 다수 제외 위험 |

**채택: 5** — 1~4회 등장 노이즈 제거, 희소 seed 7개 중 `넷제로`(7건)·`부패방지`(5건)는 min_df=5 경계에 위치 → vocab 포함 여부를 실행 후 확인

**`max_df` — 최대 등장 문서 비율**

| 옵션 | 값 | 영향 |
|---|---|---|
| 0.95 | 95% 초과 제외 | G seed(`이사회`·`사외이사`·`주주` 100%) 제외 |
| 0.80 | 80% 초과 제외 | 변별력 없는 어휘 적극 제거, G seed 핵심 3개 제외 |
| 0.99 | 99% 초과 제외 | 보일러플레이트 잔존 |

**채택: 0.80** — 2장에서 `이사회`·`사외이사`·`주주`가 381/381 등장 확인 → 변별력 없음.  
이들이 vocab에서 제외되는 것은 버그가 아닌 의도된 동작 — G cheap-talk의 핵심 증거.  
G seed 10개 중 3개만 vocab 통과 → expanded dictionary로 보강 필요성 강화

**`sublinear_tf`**

| 옵션 | 효과 |
|---|---|
| False | tf 그대로 사용 — 고빈도 단어 과대 반영 |
| **True** | 1+log(tf) — 반복 등장 단어의 과대 반응 방지 |

**채택: True** — `이사회`(평균 20.8회/문서) 같은 고빈도 seed의 과대 반영 방지

**최종 채택 파라미터**: `min_df=5, max_df=0.80, sublinear_tf=True`

In [30]:
"""
3-1. TF-IDF 학습 + Seed Score 생성

채택 파라미터 (Decision Box ⑩):
- min_df=5, max_df=0.80, sublinear_tf=True

산출:
- TFIDF_VEC, TFIDF_MAT
- PANEL_DF: seed_score_E/S/G (firm-year × 3)
"""
from sklearn.feature_extraction.text import TfidfVectorizer

# ── TF-IDF 입력 준비 ──────────────────────────────────────────────
sorted_keys    = sorted(TOKEN_DICT.keys())
corpus_strings = [" ".join(TOKEN_DICT[k]) for k in sorted_keys]

print(f"[TF-IDF 입력]")
print(f"  firm-year 수: {len(corpus_strings)}")

# ── TfidfVectorizer 학습 ──────────────────────────────────────────
TFIDF_VEC = TfidfVectorizer(
    tokenizer=lambda x: x.split(),
    token_pattern=None,
    min_df=5,
    max_df=0.80,
    sublinear_tf=True,
)
TFIDF_MAT = TFIDF_VEC.fit_transform(corpus_strings)
vocab     = TFIDF_VEC.get_feature_names_out()
vocab_set = set(vocab)
vocab_idx = {t: i for i, t in enumerate(vocab)}

print(f"\n[TF-IDF 결과]")
print(f"  행렬 shape : {TFIDF_MAT.shape}")
print(f"  vocab 크기 : {len(vocab):,}")

# ── seed vocab 포함 여부 확인 ─────────────────────────────────────
print(f"\n[seed vocab 포함 여부]")
print(f"  {'seed':<15} {'dim':<4} {'vocab':<8} {'비고'}")
print(f"  {'-'*50}")
for dim, seeds in SEED_DICT.items():
    for s in seeds:
        in_vocab = s in vocab_set
        if not in_vocab:
            # 제외 이유 진단
            n_docs = sum(1 for tokens in TOKEN_DICT.values() if s in tokens)
            pct    = n_docs / len(TOKEN_DICT) * 100
            if pct > 80:
                reason = f"max_df=0.80 초과 ({pct:.1f}% 등장) — 의도된 동작"
            elif n_docs < 5:
                reason = f"min_df=5 미달 ({n_docs}건)"
            else:
                reason = f"기타 ({n_docs}건, {pct:.1f}%)"
        else:
            reason = ""
        mark = "✓" if in_vocab else "✗"
        print(f"  {mark} {s:<15} {dim:<4} {str(in_vocab):<8} {reason}")

n_in  = sum(1 for s in ALL_SEEDS if s in vocab_set)
n_out = len(ALL_SEEDS) - n_in
print(f"\n  vocab 포함: {n_in}/30  제외: {n_out}/30")

# ── E/S/G seed score 합산 ─────────────────────────────────────────
print(f"\n[E/S/G seed score 합산]")
import scipy.sparse as sp

scores = {}
for dim, seeds in SEED_DICT.items():
    in_vocab_seeds = [s for s in seeds if s in vocab_set]
    if in_vocab_seeds:
        idx  = [vocab_idx[s] for s in in_vocab_seeds]
        col  = TFIDF_MAT[:, idx].sum(axis=1)
        scores[f"seed_score_{dim}"] = np.asarray(col).flatten()
    else:
        scores[f"seed_score_{dim}"] = np.zeros(len(sorted_keys))
    print(f"  {dim}: vocab 포함 {len(in_vocab_seeds)}/{len(seeds)}개 "
          f"→ 평균 score {scores[f'seed_score_{dim}'].mean():.4f}")

# ── PANEL_DF 초기 생성 ────────────────────────────────────────────
PANEL_DF = pd.DataFrame({
    "stock_code":  [k[0] for k in sorted_keys],
    "fiscal_year": [k[1] for k in sorted_keys],
    **scores,
    "n_tokens": [len(TOKEN_DICT[k]) for k in sorted_keys],
})
PANEL_DF["log_n_tokens"] = np.log1p(PANEL_DF["n_tokens"])

print(f"\n[PANEL_DF 초기 생성] shape={PANEL_DF.shape}")
print(PANEL_DF[["stock_code","fiscal_year",
                "seed_score_E","seed_score_S","seed_score_G"]].head(5).to_string(index=False))

[TF-IDF 입력]
  firm-year 수: 381

[TF-IDF 결과]
  행렬 shape : (381, 6811)
  vocab 크기 : 6,811

[seed vocab 포함 여부]
  seed            dim  vocab    비고
  --------------------------------------------------
  ✓ 탄소              E    True     
  ✓ 온실가스            E    True     
  ✓ 탄소중립            E    True     
  ✓ 넷제로             E    True     
  ✓ 재생에너지           E    True     
  ✓ 에너지             E    True     
  ✓ 전력              E    True     
  ✓ 폐기물             E    True     
  ✓ 재활용             E    True     
  ✓ 폐수              E    True     
  ✓ 안전              S    True     
  ✓ 산업재해            S    True     
  ✓ 중대재해            S    True     
  ✓ 임직원             S    True     
  ✓ 노동              S    True     
  ✓ 인권              S    True     
  ✓ 교육훈련            S    True     
  ✓ 협력사             S    True     
  ✓ 공급망             S    True     
  ✓ 지역사회            S    True     
  ✗ 이사회             G    False    max_df=0.80 초과 (100.0% 등장) — 의도된 동작
  ✗ 사외이사            G    False    

**TF-IDF 학습 + Seed Score 생성 완료**

**TF-IDF 행렬**: 381 × 6,811

**seed vocab 포함 결과**

| 차원 | 포함 | 제외 | 제외 seed |
|---|---|---|---|
| E | 10/10 | 0 | — |
| S | 10/10 | 0 | — |
| G | 3/10 | 7 | 이사회(100%)·사외이사(100%)·주주(100%)·감사위원회(95%)·독립성(97.6%)·준법(97.4%)·의결권(80.3%) |

- G seed 7개가 `max_df=0.80` 초과로 vocab 제외
- 이 7개는 2장에서 확인한 법정 의무 공시 어휘 — 전 문서에 등장해 변별력 없음
- G seed_score 평균 0.0086 — E(0.1130)·S(0.0667) 대비 압도적으로 낮음
- → G 차원은 seed-only 분석으로는 측정 불가 수준 → **expanded dictionary 보강이 G에서 가장 긴급**

**seed score 분포**

| 차원 | vocab 포함 | 평균 score | 해석 |
|---|---|---|---|
| E | 10/10 | 0.1130 | 정상 측정 가능 |
| S | 10/10 | 0.0667 | 정상이나 희소 seed(인권·중대재해 등) 영향으로 낮음 |
| G | 3/10 | 0.0086 | 사실상 측정 불가 — expanded 보강 필수 |

## **3-2. FastText 학습 + Expanded Dictionary**

seed-only TF-IDF의 한계를 보완하기 위해 corpus 자체로 FastText를 학습하고, seed 주변 유사어를 확장해 expanded dictionary를 구성한다.

### **Decision Box ⑪ — FastText 학습 파라미터**

| 파라미터 | 옵션 | 채택값 | 근거 |
|---|---|---|---|
| 모델 | cbow / **sg(skip-gram)** | **sg=1** | 희소 단어(넷제로·부패방지) 표현에 skip-gram이 유리 |
| 벡터 차원 | 50 / **100** / 300 | **100** | corpus 크기(381문서) 대비 적정. 300은 과적합 위험 |
| window | 3 / **5** / 10 | **5** | 한국어 사업보고서 평균 문장 길이 고려 |
| min_count | 1 / **3** / 5 | **3** | 희소 seed 보존(넷제로 9회·부패방지 8회) vs 노이즈 제거 균형 |
| epochs | 5 / **10** / 20 | **10** | 수렴 확인 후 조정 |

- skip-gram: corpus가 작을수록(381문서) 희소 단어 학습에 유리 — 희소 seed 7개 보강이 목적
- min_count=3: `넷제로`(9회)·`부패방지`(8회)가 min_count=5에서 제외될 위험 회피
- 산업특화 임베딩: 사업보고서 corpus로 직접 학습 — 범용 사전(Word2Vec 뉴스) 대비 ESG 맥락 반영

### **Decision Box ⑫ — θ (cosine threshold) Sweep**

> θ를 직접 몇 가지 값으로 바꿔가며 결과를 비교한다. 각 θ별로 상위 100단어를 훑어 확장 사전 크기와 잡음 비율을 표로 정리하고, 최종 선택한 θ에서 후보 단어를 직접 검토해 걸러낸다.

θ sweep 범위: 0.55 · 0.60 · 0.65 · 0.70 · 0.75

- θ가 낮을수록 후보 단어 수 ↑, 잡음 ↑
- θ가 높을수록 후보 단어 수 ↓, 핵심 유사어 누락 위험
- 각 θ별 후보 수·잡음 단어 비율을 표로 정리하고 최종 θ를 정당화한다

In [31]:
"""
3-2. FastText 학습 + θ Sweep

학습 파라미터 (Decision Box ⑪):
- sg=1, vector_size=100, window=5, min_count=3, epochs=10

θ Sweep (Decision Box ⑫):
- θ = 0.55 / 0.60 / 0.65 / 0.70 / 0.75
- 각 θ별 후보 수·잡음 비율 표
"""
try:
    from gensim.models import FastText as GensimFastText
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "gensim"], check=True)
    from gensim.models import FastText as GensimFastText

FT_PATH = os.path.join(OUTPUT_DIR, "fasttext_esg.model")

# ── FastText 학습 ─────────────────────────────────────────────────
if os.path.exists(FT_PATH):
    print(f"[Checkpoint] FastText 모델 존재 — 로드")
    FT_MODEL = GensimFastText.load(FT_PATH)
else:
    print(f"[FastText 학습 시작]")
    sentences = list(TOKEN_DICT.values())
    FT_MODEL  = GensimFastText(
        sentences=sentences,
        sg=1,
        vector_size=100,
        window=5,
        min_count=3,
        epochs=10,
        seed=SEED,
        workers=4,
    )
    FT_MODEL.save(FT_PATH)
    print(f"  저장: {FT_PATH}")

vocab_ft   = set(FT_MODEL.wv.key_to_index.keys())
print(f"  FastText vocab: {len(vocab_ft):,}")
print(f"\n[Sanity check — seed 유사어 Top 5]")
for seed in ["온실가스", "감사위원회", "공급망"]:
    if seed in vocab_ft:
        sims = FT_MODEL.wv.most_similar(seed, topn=5)
        print(f"  {seed}: {[w for w,_ in sims]}")

# ── θ Sweep ───────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("[θ Sweep — E/S/G 각 seed 기준 후보 수·잡음 비율]")

THETAS    = [0.55, 0.60, 0.65, 0.70, 0.75]
# 잡음 휴리스틱: 숫자 포함, 길이 1, 영문 only, 명백한 고유명사 패턴
def is_noise(word):
    if len(word) < 2:               return True
    if re.search(r"[0-9]", word):   return True
    if re.match(r"^[a-zA-Z]+$", word): return True
    return False

sweep_records = []
for theta in THETAS:
    all_cands = set()
    noise_cnt = 0
    for dim, seeds in SEED_DICT.items():
        for seed in seeds:
            if seed not in vocab_ft:
                continue
            sims = FT_MODEL.wv.most_similar(seed, topn=100)
            for word, score in sims:
                if score >= theta:
                    all_cands.add(word)
                    if is_noise(word):
                        noise_cnt += 1
    n_cands      = len(all_cands)
    noise_ratio  = noise_cnt / max(n_cands, 1) * 100
    sweep_records.append({
        "θ": theta,
        "후보 수": n_cands,
        "잡음 수(추정)": noise_cnt,
        "잡음 비율(%)": round(noise_ratio, 1),
    })
    print(f"  θ={theta}: 후보 {n_cands:>4}개  잡음 {noise_cnt:>3}개 ({noise_ratio:.1f}%)")

sweep_df = pd.DataFrame(sweep_records)
print(f"\n[Sweep 표]")
print(sweep_df.to_string(index=False))

# ── θ=0.65 상위 100단어 직접 검토 ─────────────────────────────────
print(f"\n{'='*60}")
print("[θ=0.65 — 차원별 상위 후보 30개 미리보기]")
for dim, seeds in SEED_DICT.items():
    cands_dim = {}
    for seed in seeds:
        if seed not in vocab_ft:
            continue
        sims = FT_MODEL.wv.most_similar(seed, topn=100)
        for word, score in sims:
            if score >= 0.65 and word not in ALL_SEEDS:
                if word not in cands_dim or cands_dim[word] < score:
                    cands_dim[word] = score
    top30 = sorted(cands_dim.items(), key=lambda x: -x[1])[:30]
    print(f"\n  [{dim}] {len(cands_dim)}개 후보 — 상위 30개:")
    for w, s in top30:
        noise_flag = " ←잡음?" if is_noise(w) else ""
        print(f"    {w:<15} {s:.3f}{noise_flag}")

[FastText 학습 시작]
  저장: outputs\fasttext_esg.model
  FastText vocab: 13,124

[Sanity check — seed 유사어 Top 5]
  온실가스: ['배출량', '허용량', '할당', '배출', '감축']
  감사위원회: ['감사위원', '감사실원', '감사', '위원', '감사선']
  공급망: ['교란', '각처', '선복', '지정학', '원재재']

[θ Sweep — E/S/G 각 seed 기준 후보 수·잡음 비율]
  θ=0.55: 후보 1216개  잡음  10개 (0.8%)
  θ=0.6: 후보  770개  잡음   8개 (1.0%)
  θ=0.65: 후보  493개  잡음   5개 (1.0%)
  θ=0.7: 후보  286개  잡음   5개 (1.7%)
  θ=0.75: 후보  142개  잡음   3개 (2.1%)

[Sweep 표]
   θ  후보 수  잡음 수(추정)  잡음 비율(%)
0.55  1216        10       0.8
0.60   770         8       1.0
0.65   493         5       1.0
0.70   286         5       1.7
0.75   142         3       2.1

[θ=0.65 — 차원별 상위 후보 30개 미리보기]

  [E] 123개 후보 — 상위 30개:
    배출량             0.909
    블룸버그            0.828
    허용량             0.785
    매립              0.783
    할당              0.782
    감량              0.779
    처리장             0.776
    배출              0.771
    충칭              0.757
    효율화             0.755
    재사용             0.751
    여정        

**FastText 학습 + θ Sweep 완료**

**FastText 학습 결과**
- vocab: 13,124개
- Sanity check 유사어 확인
  - `온실가스` → 배출량·허용량·할당·배출·감축 ✓ (E 의미 정합)
  - `감사위원회` → 감사위원·감사실원·감사·위원 ✓ (G 의미 정합)
  - `공급망` → 교란·각처·선복·지정학·원재재 ⚠ (지정학 리스크 맥락 노이즈 — 큐레이션 필요)

**θ Sweep 표**

| θ | 후보 수 | 잡음 수 | 잡음 비율 |
|---|---|---|---|
| 0.55 | 1,216 | 10 | 0.8% |
| 0.60 | 770 | 8 | 1.0% |
| **0.65** | **493** | **5** | **1.0%** |
| 0.70 | 286 | 5 | 1.7% |
| 0.75 | 142 | 3 | 2.1% |

**차원별 후보 특이사항 (θ=0.65)**

| 차원 | 후보 수 | 주요 관찰 |
|---|---|---|
| E | 123개 | `블룸버그`(0.828)·`충칭`(0.757)·`아너스`(0.735)·`콩기름`(0.729)·`모건스탠리`(0.723) — 금융사·지명·무관어 잡음 존재 |
| S | 29개 | `디자`(0.750)·`각처`(0.705)·`선복`(0.698)·`침공`(0.676)·`입양`(0.657) — 형태소 오분리·시사 노이즈 다수 |
| G | 329개 | 상위권에 인명(`최성락`·`이재호`·`김용` 등) 다수 — 이사·감사 관련 인명이 감사위원회 문맥에서 학습됨. `감사1팀`은 숫자 포함 잡음 |

### **Decision Box ⑫ — θ 채택 결정**

**채택: θ = 0.65**

| 기준 | θ=0.60 | **θ=0.65** | θ=0.70 |
|---|---|---|---|
| 후보 수 | 770개 | **493개** | 286개 |
| 잡음 비율 | 1.0% | **1.0%** | 1.7% |
| S 후보 수 | 충분 | **29개** | 부족 위험 |
| 검토 가능성 | 어려움 | **현실적** | 과도한 누락 |

**채택 근거**
- θ=0.60: 후보 770개 — 직접 검토 비현실적, 잡음 절대량 증가
- θ=0.65: 493개 — 차원별 100~300개 수준으로 직접 검토 가능, 잡음 비율 1.0% 유지
- θ=0.70: 286개 — S 차원 후보가 29개 이하로 급감, 이미 희소한 S 신호 추가 손실
- θ=0.75: 142개 — E·S 차원 핵심 유사어 다수 누락 위험

**비채택 근거**
- θ를 낮추면 후보 단어가 늘고 잡음도 늘어남 — θ=0.55는 1,216개로 검토 불가 수준
- 최종 선택한 θ에서 후보를 직접 검토해 걸러내는 과정이 필수

## **3-3. Expanded Dictionary 큐레이션**

### **Decision Box ⑬ — Expanded Dictionary 큐레이션 기준**

θ=0.65 후보 493개를 아래 기준으로 자동 필터 → 수동 검토 순서로 걸러낸다.

**자동 필터 (코드)**
- 길이 < 2 제거
- 영문·숫자 only 제거
- 불용어(회사명 포함) 제거
- 차원 간 중복 시 cosine 높은 쪽 유지

**수동 검토 기각 기준**
- 인명: `최성락`·`이재호`·`김용준` 등 이사·감사 인명 → **기각** (측정 대상이 아닌 주체)
- 지명·기관명: `충칭`·`모건스탠리`·`블룸버그` → **기각** (고유명사, ESG 맥락 우연)
- 형태소 오분리: `디자`(디자인 분리)·`원재재`(원재료 오분리)·`감사1팀`(숫자 포함) → **기각**
- 시사 노이즈: `침공`·`선복`·`제로코로나` → **기각** (특정 사건 맥락, 일반화 불가)
- ESG 직접 관련 어휘: `배출량`·`근로자`·`감사위원`·`내부통제` 등 → **채택**

In [32]:
"""
3-3. Expanded Dictionary 큐레이션

θ=0.65 후보 → 자동 필터 → 수동 검토 기각 → EXPANDED_DICT 확정
가이드 01 명시 요건: 채택/기각 예시 표 포함
"""

THETA = 0.65

# ── 후보 수집 ─────────────────────────────────────────────────────
cand_records = []
for dim, seeds in SEED_DICT.items():
    for seed in seeds:
        if seed not in vocab_ft:
            continue
        sims = FT_MODEL.wv.most_similar(seed, topn=100)
        for word, score in sims:
            if score >= THETA and word not in ALL_SEEDS:
                cand_records.append({
                    "dim": dim, "seed": seed,
                    "candidate": word, "cosine": round(score, 4)
                })

cand_df = pd.DataFrame(cand_records)
print(f"[θ={THETA} 후보] {len(cand_df)}행 (중복 포함)")

# ── 자동 필터 ─────────────────────────────────────────────────────
# 1) 길이 < 2
cand_df = cand_df[cand_df["candidate"].str.len() >= 2]
# 2) 영문·숫자 only
hangul = re.compile(r"[가-힣]")
cand_df = cand_df[cand_df["candidate"].apply(lambda x: bool(hangul.search(x)))]
# 3) 불용어
cand_df = cand_df[~cand_df["candidate"].isin(STOPWORDS_RAW)]
# 4) 차원 간 중복 — cosine 높은 쪽 유지
cand_df = (cand_df.sort_values("cosine", ascending=False)
           .drop_duplicates(subset=["candidate"]).copy().reset_index(drop=True))

print(f"  자동 필터 후: {len(cand_df)}개")

# ── 수동 검토 기각 목록 ───────────────────────────────────────────
REJECT = {
    # 인명
    "최성락","이재호","김용","최원욱","임성균","최영주","윤영선","이성락",
    "최원준","김문수","이재진","노혁준","김용준","퇴임후보","이복영",
    "유정준","김준기","이성래","황윤철","최준기","박재하","최원일",
    "곽봉환","한성희","최윤정","이동현","김재성","정도진","이재명",
    # 지명·기관명
    "충칭","모건스탠리","블룸버그","태화강","각처",
    # 형태소 오분리·오타
    "디자","원재재","감사선","감사실원","감사1팀","녹생","메뉴얼",
    # 시사 노이즈
    "침공","선복","제로코로나","발발","병목교란","각처","지정학",
    # 의미 무관
    "아너스","콩기름","나노급","패인","입성","종말","여정","입양","웨이스트",
}

cand_df["rejected"] = cand_df["candidate"].isin(REJECT)
n_reject = cand_df["rejected"].sum()
print(f"  수동 기각: {n_reject}개")

EXPANDED_CAND_DF = cand_df[~cand_df["rejected"]].copy().reset_index(drop=True)
print(f"  최종 채택 후보: {len(EXPANDED_CAND_DF)}개")

# ── EXPANDED_DICT 구축 ────────────────────────────────────────────
EXPANDED_DICT = {dim: list(seeds) for dim, seeds in SEED_DICT.items()}
for _, row in EXPANDED_CAND_DF.iterrows():
    EXPANDED_DICT[row["dim"]].append(row["candidate"])
for dim in EXPANDED_DICT:
    EXPANDED_DICT[dim] = sorted(set(EXPANDED_DICT[dim]))

print(f"\n[EXPANDED_DICT 최종]")
for dim in ["E","S","G"]:
    print(f"  {dim}: {len(EXPANDED_DICT[dim])}개 (seed {len(SEED_DICT[dim])}개 포함)")

# ── 채택/기각 예시 표 (가이드 01 명시 요건) ───────────────────────
print(f"\n{'='*60}")
print("[채택/기각 예시 표 — 보고서용]")
manual_examples = [
    # (후보, seed, 결정, 사유)
    ("배출량",    "온실가스", "채택", "E 측정 단위, 의미 직접 연결"),
    ("감축",      "탄소",    "채택", "E 감소 행위, seed 보완"),
    ("재사용",    "재활용",  "채택", "E 동의어 계열"),
    ("근로자",    "노동",    "채택", "S 직접 관련 (근로기준법 맥락)"),
    ("보건",      "안전",    "채택", "S 안전보건 묶음 어휘"),
    ("감사위원",  "감사위원회","채택","G 직접 관련 (위원회 구성원)"),
    ("부패",      "부패방지", "채택", "G seed 보완 어휘"),
    ("블룸버그",  "온실가스", "기각", "금융정보기관 고유명사, ESG 맥락 우연"),
    ("디자",      "인권",    "기각", "'디자인' 형태소 오분리"),
    ("침공",      "공급망",  "기각", "우크라이나 침공 시사 노이즈"),
    ("최성락",    "감사위원회","기각","이사 인명, 측정 대상 아님"),
    ("콩기름",    "재생에너지","기각","의미 무관 (우연한 문맥 유사도)"),
]

ex_df = pd.DataFrame(manual_examples,
                     columns=["후보","seed","결정","사유"])
# 실제 cosine 값 채우기
def get_cosine(cand, seed_):
    match = cand_df[
        (cand_df["candidate"]==cand) & (cand_df["seed"]==seed_)
    ]
    return round(match.iloc[0]["cosine"], 3) if len(match) > 0 else None

ex_df["cosine"] = ex_df.apply(lambda r: get_cosine(r["후보"], r["seed"]), axis=1)
ex_df = ex_df[["후보","seed","cosine","결정","사유"]]
print(ex_df.to_string(index=False))

# 저장
ex_df.to_csv(os.path.join(OUTPUT_DIR,"expanded_review_examples.csv"),
             index=False, encoding="utf-8-sig")
print(f"\n[저장] expanded_review_examples.csv")

[θ=0.65 후보] 660행 (중복 포함)
  자동 필터 후: 475개
  수동 기각: 42개
  최종 채택 후보: 433개

[EXPANDED_DICT 최종]
  E: 121개 (seed 10개 포함)
  S: 26개 (seed 10개 포함)
  G: 316개 (seed 10개 포함)

[채택/기각 예시 표 — 보고서용]
  후보  seed  cosine 결정                     사유
 배출량  온실가스   0.909 채택      E 측정 단위, 의미 직접 연결
  감축    탄소     NaN 채택       E 감소 행위, seed 보완
 재사용   재활용   0.751 채택               E 동의어 계열
 근로자    노동     NaN 채택     S 직접 관련 (근로기준법 맥락)
  보건    안전   0.768 채택           S 안전보건 묶음 어휘
감사위원 감사위원회   0.973 채택      G 직접 관련 (위원회 구성원)
  부패  부패방지   0.887 채택           G seed 보완 어휘
블룸버그  온실가스     NaN 기각 금융정보기관 고유명사, ESG 맥락 우연
  디자    인권   0.750 기각          '디자인' 형태소 오분리
  침공   공급망   0.676 기각        우크라이나 침공 시사 노이즈
 최성락 감사위원회     NaN 기각        이사 인명, 측정 대상 아님
 콩기름 재생에너지     NaN 기각     의미 무관 (우연한 문맥 유사도)

[저장] expanded_review_examples.csv


**Expanded Dictionary 큐레이션 완료**

**후보 처리 흐름**

| 단계 | 개수 |
|---|---|
| θ=0.65 초기 후보 (중복 포함) | 660개 |
| 자동 필터 후 | 475개 |
| 수동 기각 후 | **433개** |

**EXPANDED_DICT 최종 구성**

| 차원 | seed | 확장 후보 | 합계 |
|---|---|---|---|
| E | 10개 | 111개 | **121개** |
| S | 10개 | 16개 | **26개** |
| G | 10개 | 306개 | **316개** |

- **S 차원 확장 후보 16개** — 2장에서 예고한 seed 희소성이 확장에서도 나타남. S 신호 자체가 본질적으로 약함을 확인
- **G 차원 확장 후보 306개** — seed 7개가 max_df로 제외됐음에도 감사위원·부패·감시인·내부통제 등 실천 어휘로 대폭 보강
- `감축`·`근로자`·`블룸버그`·`최성락` 등 cosine=NaN은 θ=0.65 후보에 없거나 자동 필터에서 제거된 것 — 기각 판단에 영향 없음

**채택/기각 예시 (대표 12개)**

| 후보 | seed | cosine | 결정 | 사유 |
|---|---|---|---|---|
| 배출량 | 온실가스 | 0.909 | **채택** | E 측정 단위, 의미 직접 연결 |
| 재사용 | 재활용 | 0.751 | **채택** | E 동의어 계열 |
| 보건 | 안전 | 0.768 | **채택** | S 안전보건 묶음 어휘 |
| 감사위원 | 감사위원회 | 0.973 | **채택** | G 직접 관련 (위원회 구성원) |
| 부패 | 부패방지 | 0.887 | **채택** | G seed 보완 어휘 |
| 블룸버그 | 온실가스 | — | **기각** | 금융정보기관 고유명사 |
| 디자 | 인권 | 0.750 | **기각** | '디자인' 형태소 오분리 |
| 침공 | 공급망 | 0.676 | **기각** | 시사 노이즈 |
| 최성락 | 감사위원회 | — | **기각** | 이사 인명, 측정 대상 아님 |
| 콩기름 | 재생에너지 | — | **기각** | 의미 무관 |

## **3-4. Expanded Score + 기준 문장 Cosine**

### **Expanded TF-IDF Score**

seed score와 동일한 방식으로 EXPANDED_DICT 기준 TF-IDF 값을 합산한다.

- 계산식은 seed score와 동일 — 합산할 단어 집합만 커짐
- seed score와 expanded score를 동시에 같은 회귀식에 투입하지 않는다 — 다중공선성 위험
- 회귀에서는 feature set별로 분리해 비교 (M1a seed / M1b expanded / M1c cosine)

### **Decision Box ⑭ — 기준 문장 Cosine Similarity**

| 옵션 | 방식 | 한계 |
|---|---|---|
| TF-IDF 기반 cosine | 기준 문장 ↔ 공시 문장 TF-IDF 벡터 방향 비교 | 단어 겹침 기반, 의미 유사도 아님 |
| Dense vector cosine | 문장 임베딩 기반 | 심화 분석 수준, 본선 범위 초과 |

**채택: TF-IDF 기반 cosine (보조 측정)**

- 단어 겹침 기반이라 embedding 의미 유사도와 다름. 비교용 baseline으로 해석
- seed/expanded score와 독립적인 보조 측정값으로 활용
- 절대값보다 firm-year 간 상대적 ranking에 의미 있음

**기준 문장**

- E: "당사는 온실가스 배출량을 감축하고 재생에너지 사용을 확대하였다"
- S: "협력사의 안전보건 교육을 강화하고 공급망 점검을 확대하였다"
- G: "이사회 내 감사위원회의 독립성과 내부통제 절차를 강화하였다"

### **다중공선성 — Decision Box ⑮**

seed/expanded/cosine 세 feature set을 동시에 회귀에 투입하면 다중공선성 문제 발생.

**feature set별 분리 회귀**
- M1a: seed score E/S/G + log_n_tokens
- M1b: expanded score E/S/G + log_n_tokens
- M1c: cosine E/S/G + log_n_tokens
- 같은 차원 내 seed↔expanded 상관이 매우 높을 것으로 예상 (3장 결론에서 확인)
- G 차원은 seed↔expanded 상관이 낮을 가능성 — seed_G의 vocab 제외 때문

In [ ]:
"""
3-4. Expanded Score + 기준 문장 Cosine + PANEL_DF 완성

- expanded_score_E/S/G: EXPANDED_DICT 기준 TF-IDF 합산
- ref_cosine_E/S/G: 기준 문장 ↔ 공시 문장 TF-IDF cosine
- 다중공선성 진단 (feature 간 상관)
- PANEL_DF 최종 완성
"""
from sklearn.metrics.pairwise import cosine_similarity

# ── Expanded Score ────────────────────────────────────────────────
print("[Expanded Score 계산]")
for dim in ["E","S","G"]:
    in_vocab = [w for w in EXPANDED_DICT[dim] if w in vocab_set]
    if in_vocab:
        idx  = [vocab_idx[w] for w in in_vocab]
        col  = TFIDF_MAT[:, idx].sum(axis=1)
        PANEL_DF[f"expanded_score_{dim}"] = np.asarray(col).flatten()
    else:
        PANEL_DF[f"expanded_score_{dim}"] = 0.0
    n_in = len(in_vocab)
    n_total_exp = len(EXPANDED_DICT[dim])
    mean_sc = PANEL_DF[f"expanded_score_{dim}"].mean()
    print(f"  {dim}: vocab 포함 {n_in}/{n_total_exp}개 → 평균 score {mean_sc:.4f}")

# ── 기준 문장 Cosine ──────────────────────────────────────────────
print(f"\n[기준 문장 Cosine 계산]")
ref_sentences = {
    "E": "당사는 온실가스 배출량을 감축하고 재생에너지 사용을 확대하였다",
    "S": "협력사의 안전보건 교육을 강화하고 공급망 점검을 확대하였다",
    "G": "이사회 내 감사위원회의 독립성과 내부통제 절차를 강화하였다",
}

for dim, sent in ref_sentences.items():
    tokens   = extract_nouns_kiwi(sent, kiwi)
    filtered = [t for t in tokens if t not in STOPWORDS]
    ref_str  = " ".join(filtered)
    ref_vec  = TFIDF_VEC.transform([ref_str])
    sims     = cosine_similarity(TFIDF_MAT, ref_vec).flatten()
    PANEL_DF[f"ref_cosine_{dim}"] = sims
    print(f"  {dim} 기준 문장 토큰: {filtered}")
    print(f"     cosine 평균: {sims.mean():.4f}  최대: {sims.max():.4f}")

# ── 다중공선성 진단 ───────────────────────────────────────────────
print(f"\n{'='*60}")
print("[다중공선성 진단 — feature 간 Pearson 상관]")
feature_cols = [
    "seed_score_E","seed_score_S","seed_score_G",
    "expanded_score_E","expanded_score_S","expanded_score_G",
    "ref_cosine_E","ref_cosine_S","ref_cosine_G",
    "log_n_tokens",
]
corr_mat = PANEL_DF[feature_cols].corr().round(3)
print(corr_mat.to_string())

print(f"\n[같은 차원 내 seed↔expanded 상관]")
for dim in ["E","S","G"]:
    r = PANEL_DF[f"seed_score_{dim}"].corr(PANEL_DF[f"expanded_score_{dim}"])
    print(f"  {dim}: seed↔expanded r={r:.3f}  "
          f"→ {'동시 투입 금지' if abs(r)>0.8 else '동시 투입 가능'}")

# ── PANEL_DF 최종 저장 ────────────────────────────────────────────
panel_path = os.path.join(OUTPUT_DIR, "analysis_panel.csv")
PANEL_DF.to_csv(panel_path, index=False, encoding="utf-8-sig")

print(f"\n[PANEL_DF 최종] shape={PANEL_DF.shape}")
print(f"  columns: {list(PANEL_DF.columns)}")
print(PANEL_DF[["stock_code","fiscal_year",
                "seed_score_E","expanded_score_E","ref_cosine_E"]].head(5).to_string(index=False))
print(f"\n[저장] {panel_path}")

[Expanded Score 계산]
  E: vocab 포함 78/121개 → 평균 score 0.3330
  S: vocab 포함 24/26개 → 평균 score 0.1090
  G: vocab 포함 70/316개 → 평균 score 0.1965

[기준 문장 Cosine 계산]
  E 기준 문장 토큰: ['온실가스', '배출량', '감축', '재생에너지']
     cosine 평균: 0.0239  최대: 0.1222
  S 기준 문장 토큰: ['협력사', '안전', '보건', '교육', '공급망', '점검']
     cosine 평균: 0.0221  최대: 0.1047
  G 기준 문장 토큰: ['이사회', '감사위원회', '독립성', '내부', '통제']
     cosine 평균: 0.0160  최대: 0.0828

[다중공선성 진단 — feature 간 Pearson 상관]
                  seed_score_E  seed_score_S  seed_score_G  expanded_score_E  expanded_score_S  expanded_score_G  ref_cosine_E  ref_cosine_S  ref_cosine_G  log_n_tokens
seed_score_E             1.000         0.252         0.035             0.937             0.220            -0.029         0.834         0.240         0.006         0.321
seed_score_S             0.252         1.000         0.081             0.366             0.817             0.007         0.212         0.650        -0.193         0.377
seed_score_G             0.035         0.081   

**PANEL_DF 완성 — Feature 생성 완료**

**Expanded Score 결과**

| 차원 | vocab 포함 | 평균 score | seed 대비 변화 |
|---|---|---|---|
| E | 78/121 (64%) | 0.3330 | seed 0.1130 → **+0.2200 (+195%)** |
| S | 24/26 (92%) | 0.1090 | seed 0.0667 → **+0.0423 (+63%)** |
| G | 70/316 (22%) | 0.1965 | seed 0.0086 → **+0.1879 (+2185%)** |

- G 차원에서 expanded dictionary의 정당성이 가장 강함 — seed vocab 제외를 expanded가 대폭 보완
- G의 vocab 포함율 22%(70/316)이 낮은 이유: 큐레이션 채택 후보 중 min_df=5 미달 단어가 많음 — G 관련 세부 어휘의 희소성 반영

**기준 문장 Cosine 결과**

| 차원 | 평균 cosine | 최대 cosine |
|---|---|---|
| E | 0.0239 | 0.1222 |
| S | 0.0221 | 0.1047 |
| G | 0.0160 | 0.0828 |

- 절대값이 전반적으로 낮음 — TF-IDF 기반 cosine의 본질적 한계 (기준 문장이 짧아 벡터가 희소)
- 보조 feature로만 활용, 단독 해석 금지
- G cosine이 가장 낮음 — G 기준 문장의 핵심어(`이사회`·`감사위원회`)가 max_df로 vocab 제외된 영향

**다중공선성 진단**

| 차원 | seed↔expanded 상관 | 판정 |
|---|---|---|
| E | **r=0.937** | 동시 투입 금지 |
| S | **r=0.817** | 동시 투입 금지 |
| G | r=0.177 | 동시 투입 가능 |

- E·S는 seed와 expanded가 사실상 같은 정보를 측정 → 회귀에서 feature set별 분리 필수
- G만 seed↔expanded 상관이 낮음 — seed_G가 vocab 제외로 거의 0인 반면 expanded_G는 독립적 신호 보유. G에서는 joint model 알파 분석 가능

**추가 패턴**

- `seed_score_E` ↔ `ref_cosine_E` r=0.834 — E seed와 E cosine도 동시 투입 금지
- `ref_cosine_G` ↔ `expanded_score_G` r=0.399 — G에서는 cosine과 expanded가 상대적으로 독립적
- `log_n_tokens` ↔ 모든 feature 상관 0.16~0.42 — cheap-talk 통제 변수로 모든 회귀에 투입 필요

## **3-5. Feature 해석 정합성 — 상위/하위 문장 직접 읽기**

회귀 전에 feature 점수가 높은 firm-year와 낮은 firm-year의 실제 텍스트를 직접 읽어 feature가 의도한 신호를 포착하고 있는지 정성 검증한다.

**검증 대상**: `expanded_score_E` (E 차원 대표), `expanded_score_G` (G 차원 대표)

**기대 패턴**
- 상위 firm-year: 해당 차원 관련 어휘가 풍부하게 등장
- 하위 firm-year: 해당 차원 어휘가 거의 없거나 형식적 언급에 그침

In [ ]:
"""
3-5. Feature 해석 정합성 — 상위/하위 문장 직접 읽기

expanded_score_E, expanded_score_G 상위/하위 5개 firm-year
원문에서 관련 어휘 포함 문장 발췌 후 직접 확인
"""

def find_keyword_sentences(text, keywords, max_n=3, min_len=30, max_len=300):
    """텍스트에서 keyword 포함 문장 추출."""
    sentences = re.split(r"[.!?\n]+", text)
    matched   = []
    for s in sentences:
        s = s.strip()
        if not (min_len <= len(s) <= max_len):
            continue
        if any(kw in s for kw in keywords):
            matched.append(s)
            if len(matched) >= max_n:
                break
    return matched

# company_name 조회용
name_map = (META_INPUT_DF[["stock_code","company_name"]]
            .drop_duplicates()
            .set_index("stock_code")["company_name"].to_dict())

e_keywords = SEED_DICT["E"] + ["배출량","감축","이산화탄소","탄소배출","재생","저탄소"]
g_keywords = ["감사위원회","내부통제","감사위원","독립성","사외이사","이사회","준법"]

for feat, kws, label in [
    ("expanded_score_E", e_keywords, "E"),
    ("expanded_score_G", g_keywords, "G"),
]:
    print(f"\n{'='*60}")
    print(f"[{feat} 상위/하위 5개 firm-year]")

    merged = PANEL_DF[["stock_code","fiscal_year",feat]].copy()
    merged["company_name"] = merged["stock_code"].map(name_map)

    top5 = merged.nlargest(5, feat)
    bot5 = merged.nsmallest(5, feat)

    print(f"\n  [상위 5]")
    print(top5[["company_name","stock_code","fiscal_year",feat]].to_string(index=False))
    print(f"\n  [하위 5]")
    print(bot5[["company_name","stock_code","fiscal_year",feat]].to_string(index=False))

    # 상위 1개 원문 발췌
    top1   = top5.iloc[0]
    top_key = (top1["stock_code"], int(top1["fiscal_year"]))
    sents  = find_keyword_sentences(CORPUS_DICT[top_key], kws, max_n=3)
    print(f"\n  [상위 1위 원문 발췌 — {top1['company_name']} FY{int(top1['fiscal_year'])}]")
    print(f"  {feat}={top1[feat]:.4f}")
    if sents:
        for i, s in enumerate(sents, 1):
            print(f"    [{i}] {s[:250]}")
    else:
        print(f"    (키워드 포함 문장 없음)")

    # 하위 1개 원문 발췌
    bot1    = bot5.iloc[0]
    bot_key = (bot1["stock_code"], int(bot1["fiscal_year"]))
    sents   = find_keyword_sentences(CORPUS_DICT[bot_key], kws, max_n=3)
    print(f"\n  [하위 1위 원문 발췌 — {bot1['company_name']} FY{int(bot1['fiscal_year'])}]")
    print(f"  {feat}={bot1[feat]:.4f}")
    if sents:
        for i, s in enumerate(sents, 1):
            print(f"    [{i}] {s[:250]}")
    else:
        print(f"    (키워드 포함 문장 없음 — feature 해석 일관)")

print(f"\n[3장 완료] PANEL_DF shape={PANEL_DF.shape}")


[expanded_score_E 상위/하위 5개 firm-year]

  [상위 5]
company_name stock_code  fiscal_year  expanded_score_E
   아모레퍼시픽홀딩스     002790         2022          1.718342
       깨끗한나라     004540         2024          1.634616
      SGC에너지     005090         2023          1.514687
      SGC에너지     005090         2022          1.487256
      SGC에너지     005090         2024          1.447112

  [하위 5]
company_name stock_code  fiscal_year  expanded_score_E
        유유제약     000220         2022               0.0
        유유제약     000220         2023               0.0
        유유제약     000220         2024               0.0
       삼천당제약     000250         2022               0.0
       삼천당제약     000250         2023               0.0

  [상위 1위 원문 발췌 — 아모레퍼시픽홀딩스 FY2022]
  expanded_score_E=1.7183
    [1] 한편, 연결회사는 내부 자금 공유 확대를 통한 외부차입 최소화, 고금리 차입금 감축, 장/단기 차입구조 개선, 고정 대 변동이자 차입조건의 적정비율 유지, 일간/주간/월간 단위의 국내외 금리동향 모니터링 실시 및 대응방안 수립 및 변동금리부 조건의 단기차입금과 예금을 적절히 운영함으로써 이자율변동에 따른 위험을 최소화하고 있습니다
    [2] (4) 물류, 폐기물 관리를 포

**Feature 해석 정합성 — 상위/하위 문장 직접 읽기 완료**

**expanded_score_E 검증**

| 구분 | 기업 | FY | 점수 | 원문 확인 |
|---|---|---|---|---|
| 상위 1위 | 아모레퍼시픽홀딩스 | 2022 | 1.718 | 폐기물 관리·사업 전생애주기 환경보호·에너지/수자원 재사용·재활용 서술 직접 확인 ✓ |
| 하위 1위 | 유유제약 | 2022 | 0.000 | E 키워드 포함 문장 없음 — feature 해석 일관 ✓ |

- 상위 firm-year에서 E 어휘(`폐기물`, `재사용`, `재활용`, `에너지`)가 풍부하게 등장
- 하위 firm-year(유유제약 3개 연도 모두 0.000)는 제약업 특성상 환경 공시가 거의 없음 — 업종 효과 존재, 6장 알파에서 업종 FE 통제 필요성 확인

**expanded_score_G 검증**

| 구분 | 기업 | FY | 점수 | 원문 확인 |
|---|---|---|---|---|
| 상위 1위 | 삼성SDI | 2023 | 0.682 | 이사회 승인·준법경영·이사회 구성(사내이사 2인·사외이사 4인) 서술 직접 확인 ✓ |
| 하위 1위 | KR모터스 | 2024 | 0.000 | 이사회·감사위원회 언급 있으나 expanded G 어휘(`감사위원`·`내부통제` 등) 미포함 |

- 하위 1위(KR모터스) 해석 주의: 점수가 0이지만 원문에 "이사회", "감사위원회" 표현은 존재 — 이는 이들이 `max_df=0.80` 초과로 vocab에서 제외됐기 때문
- expanded_G가 포착하는 것은 의무 공시 어휘가 아닌 실천 어휘의 차별적 사용 여부 — "별도의 감사위원회를 설치하지 않음"처럼 형식적 언급은 점수화되지 않음
- 이 패턴이 G cheap-talk 분석의 핵심 근거

**결론**

- `PANEL_DF` 381 × 13 완성 (`outputs/analysis_panel.csv` 저장)
- 9개 feature(E·S·G × seed/expanded/cosine) + `n_tokens` + `log_n_tokens`
- feature 해석 정합성 정성 검증 완료

---

# **4. Validity 검증 — Spearman · Mann-Whitney**

## **4-1. 검증 목적**

- feature 분포만 보고 회귀로 직행하면 등급과 무관한 변수를 모형에 포함할 위험
- 서열변수(ESG 등급)에 Pearson이 아닌 Spearman 순위상관이 적합
- Spearman에 더해 Mann-Whitney U로 상위/하위 등급 그룹 간 분포 차이도 확인

### **Decision Box ⑯ — Validity 검증 방법**

**검증 설계**
- 등급 4종 × feature 10개(9개 + n_tokens) = 40개 Spearman 검증
- 상위 등급(A 이상, esg_grade_num ≥ 4) vs 하위 등급(B+ 이하, ≤ 3) Mann-Whitney
- n_tokens를 반드시 포함 — "total_word_count가 최강이라는 점을 숨기면 안 됨" (대시보드 21단계)
- 효과크기 해석 기준: |ρ| < 0.1 무의미 · 0.1~0.3 약함 · 0.3~0.5 중간 · ≥0.5 강함

### **ESG 등급 숫자 변환**

- A+ = 5
- A = 4
- B+ = 3
- B = 2
- C = 1
- D = 0

In [ ]:
"""
4. Validity 검증

① ESG 등급 숫자화 + PANEL_DF join
② Spearman 순위상관 40개 (9 feature + n_tokens × 4 grade)
③ Mann-Whitney U + 효과크기 + bootstrap CI
"""
from scipy.stats import spearmanr, mannwhitneyu

# ── ① 등급 숫자화 + join ──────────────────────────────────────────
GRADE_MAP  = {"S":6,"A+":5,"A":4,"B+":3,"B":2,"C":1,"D":0}
grade_cols = ["esg_grade","e_grade","s_grade","g_grade"]

meta_grades = META_INPUT_DF[["stock_code","fiscal_year"] + grade_cols].copy()
meta_grades["stock_code"]  = meta_grades["stock_code"].astype(str).str.zfill(6)
meta_grades["fiscal_year"] = meta_grades["fiscal_year"].astype(int)
for c in grade_cols:
    meta_grades[f"{c}_num"] = meta_grades[c].map(GRADE_MAP)

PANEL_DF["fiscal_year"] = PANEL_DF["fiscal_year"].astype(int)
ANALYSIS_DF = PANEL_DF.merge(
    meta_grades, on=["stock_code","fiscal_year"],
    how="left", validate="one_to_one"
)

print(f"[ANALYSIS_DF join] shape={ANALYSIS_DF.shape}")
print(f"\n[등급 분포]")
for c in grade_cols:
    counts = ANALYSIS_DF[c].value_counts().sort_index()
    print(f"  {c}: {dict(counts)}")

# ── ② Spearman 40개 ───────────────────────────────────────────────
feature_cols = [
    "seed_score_E","seed_score_S","seed_score_G",
    "expanded_score_E","expanded_score_S","expanded_score_G",
    "ref_cosine_E","ref_cosine_S","ref_cosine_G",
    "n_tokens",
]
grade_num_cols = ["esg_grade_num","e_grade_num","s_grade_num","g_grade_num"]

print(f"\n{'='*60}")
print("[② Spearman 순위상관 — feature × grade (40개)]")

sp_records = []
for feat in feature_cols:
    for grade in grade_num_cols:
        rho, p = spearmanr(
            ANALYSIS_DF[feat], ANALYSIS_DF[grade], nan_policy="omit"
        )
        sp_records.append({
            "feature": feat, "grade": grade,
            "rho": round(rho,3), "p_value": round(p,4),
            "abs_rho": round(abs(rho),3),
        })

sp_df = pd.DataFrame(sp_records)

# pivot
pivot = sp_df.pivot(index="feature", columns="grade", values="rho")
pivot = pivot.reindex(feature_cols)
print(f"\n  [Spearman ρ — feature × grade]")
print(pivot.round(3).to_string())

print(f"\n  [상위 15 (|ρ| 기준)]")
top15 = sp_df.sort_values("abs_rho", ascending=False).head(15)
print(top15[["feature","grade","rho","p_value"]].to_string(index=False))

# cheap-talk 경고
nt_max = sp_df[sp_df["feature"]=="n_tokens"]["abs_rho"].max()
esg_max = sp_df[sp_df["feature"]!="n_tokens"]["abs_rho"].max()
print(f"\n  [Cheap-talk 진단]")
print(f"  n_tokens 최대 |ρ|    : {nt_max:.3f}")
print(f"  ESG feature 최대 |ρ|: {esg_max:.3f}")
if nt_max > esg_max:
    print(f"  ⚠ n_tokens가 ESG feature보다 강함 — cheap-talk 위험 큼")

# 저장
sp_df.to_csv(os.path.join(OUTPUT_DIR,"spearman_results.csv"),
             index=False, encoding="utf-8-sig")

# ── ③ Mann-Whitney U ──────────────────────────────────────────────
print(f"\n{'='*60}")
print("[③ Mann-Whitney U — A 이상 vs B+ 이하]")

high_mask = ANALYSIS_DF["esg_grade_num"] >= 4
low_mask  = ANALYSIS_DF["esg_grade_num"] <= 3
print(f"  상위(A 이상) : {high_mask.sum()}건")
print(f"  하위(B+ 이하): {low_mask.sum()}건")

mw_records = []
for feat in feature_cols:
    h = ANALYSIS_DF.loc[high_mask, feat].values
    l = ANALYSIS_DF.loc[low_mask,  feat].values
    stat, p   = mannwhitneyu(h, l, alternative="two-sided")
    r_rb      = 1 - (2 * stat) / (high_mask.sum() * low_mask.sum())
    mean_diff = h.mean() - l.mean()

    # bootstrap CI
    np.random.seed(SEED)
    boots = [
        np.random.choice(h, len(h), replace=True).mean() -
        np.random.choice(l, len(l), replace=True).mean()
        for _ in range(1000)
    ]
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])

    def effect_label(r):
        a = abs(r)
        if a < 0.1:   return "무의미"
        elif a < 0.3: return "약함"
        elif a < 0.5: return "중간"
        else:         return "강함"

    mw_records.append({
        "feature": feat,
        "mean_high": round(h.mean(),4), "mean_low": round(l.mean(),4),
        "mean_diff": round(mean_diff,4),
        "p_value":   round(p,4),
        "r_rb":      round(r_rb,3),
        "effect":    effect_label(r_rb),
        "ci_lo":     round(ci_lo,4), "ci_hi": round(ci_hi,4),
    })

mw_df = pd.DataFrame(mw_records).sort_values("r_rb", ascending=False)
print(mw_df[["feature","mean_high","mean_low",
             "p_value","r_rb","effect"]].to_string(index=False))

print(f"\n  [Bootstrap 95% CI — mean_diff]")
print(mw_df[["feature","mean_diff","ci_lo","ci_hi"]].to_string(index=False))

mw_df.to_csv(os.path.join(OUTPUT_DIR,"mannwhitney_results.csv"),
             index=False, encoding="utf-8-sig")
print(f"\n[저장] spearman_results.csv, mannwhitney_results.csv")
print(f"[ANALYSIS_DF] shape={ANALYSIS_DF.shape}")

[ANALYSIS_DF join] shape=(381, 21)

[등급 분포]
  esg_grade: {'A': np.int64(115), 'A+': np.int64(23), 'B': np.int64(38), 'B+': np.int64(77), 'C': np.int64(57), 'D': np.int64(71)}
  e_grade: {'A': np.int64(113), 'A+': np.int64(30), 'B': np.int64(32), 'B+': np.int64(71), 'C': np.int64(78), 'D': np.int64(57)}
  s_grade: {'A': np.int64(95), 'A+': np.int64(112), 'B': np.int64(19), 'B+': np.int64(48), 'C': np.int64(34), 'D': np.int64(73)}
  g_grade: {'A': np.int64(76), 'A+': np.int64(18), 'B': np.int64(57), 'B+': np.int64(104), 'C': np.int64(60), 'D': np.int64(66)}

[② Spearman 순위상관 — feature × grade (40개)]

  [Spearman ρ — feature × grade]
grade             e_grade_num  esg_grade_num  g_grade_num  s_grade_num
feature                                                               
seed_score_E            0.386          0.306        0.235        0.329
seed_score_S            0.233          0.198        0.158        0.195
seed_score_G            0.196          0.248        0.271        0.170
expand

**Validity 검증 완료**

**등급 분포 확인**

| 등급 | esg | e | s | g |
|---|---|---|---|---|
| A+ | 23 | 30 | 112 | 18 |
| A | 115 | 113 | 95 | 76 |
| B+ | 77 | 71 | 48 | 104 |
| B | 38 | 32 | 19 | 57 |
| C | 57 | 78 | 34 | 60 |
| D | 71 | 57 | 73 | 66 |

- s_grade에서 A+(112건)가 압도적으로 많음

**② Spearman ρ 결과**

| feature | esg | e | s | g | 해석 |
|---|---|---|---|---|---|
| **n_tokens** | **0.663** | **0.657** | **0.631** | **0.605** | **압도적 1위 — cheap-talk 핵심 증거** |
| expanded_score_E | 0.373 | 0.451 | 0.397 | 0.299 | 중간 연관, E feature가 e_grade에 가장 강함 |
| expanded_score_G | 0.425 | 0.364 | 0.383 | 0.426 | 중간 연관, g_grade에 가장 강함 (예상과 일치) |
| seed_score_E | 0.306 | 0.386 | 0.329 | 0.235 | 약한~중간 연관 |
| expanded_score_S | 0.212 | 0.248 | 0.230 | 0.188 | 약한 연관 |
| seed_score_G | 0.248 | 0.196 | 0.170 | 0.271 | 약한 연관 — seed vocab 제외 영향 |
| ref_cosine_G | 0.248 | 0.152 | 0.234 | 0.290 | 약한 연관 |

**Cheap-talk 진단**
- `n_tokens` 최대 |ρ| = **0.663** > ESG feature 최대 |ρ| = 0.451
- 분량이 어떤 ESG 어휘 feature보다 등급과 강하게 연관 — 회귀에서 `log_n_tokens` 통제 필수

**③ Mann-Whitney U 결과 (A 이상 138건 vs B+ 이하 243건)**

| feature | 효과크기 | p-value | 해석 |
|---|---|---|---|
| **n_tokens** | **강함 (r=−0.649)** | <0.001 | A 이상 평균 7,523토큰 vs B+ 이하 3,547토큰 — **2.1배 차이** |
| expanded_score_E | 중간 (r=−0.330) | <0.001 | A 이상 0.442 vs B+ 이하 0.271 |
| expanded_score_G | 중간 (r=−0.383) | <0.001 | A 이상 0.241 vs B+ 이하 0.171 |
| 나머지 ESG feature | 약함 | <0.01 | 통계적으로 유의하나 효과 작음 |

- r 부호가 음수인 이유: Mann-Whitney U 계산 방식상 high 그룹이 low 그룹보다 크면 음수 — **방향은 상위 등급일수록 feature 값이 높음으로 해석**
- `expanded_score_S` 95% CI가 0을 포함(−0.0018~0.0382) — S 차원 평균차는 통계적으로 불안정
- 모든 feature의 95% CI가 0 포함하지 않음 (expanded_score_S 제외) — 평균차는 유의하나 크기는 작음

**결론**

- 9개 ESG feature 모두 p<0.01로 등급과 통계적으로 연관 — validity gate 통과
- 효과크기는 모두 약함~중간 수준 — 별표가 많아도 실질 효과가 큰 것 아님
- n_tokens가 압도적 1위(ρ=0.663, 효과크기 강함) — 회귀에서 cheap-talk 통제 필수
- S 차원 신호가 전반적으로 가장 약함 — corpus 희소성의 본질적 한계 재확인

---

# **5. 회귀 분석 3종 — OLS · Ordered Logistic · Binary Logistic**

## **5-1. 모형 선택 설계**

ESG feature와 등급의 연관을 회귀 모형으로 정량화한다.

### **Decision Box ⑰ — 모형 선택**

| 모형 | 종속변수 처리 | 가정 | 장점 | 한계 |
|---|---|---|---|---|
| **M1a OLS** | 연속변수처럼 | 등급 간 등간성 (D=0~S=6 균등 간격) | 계수 해석 직관적, baseline | 서열변수 등간성 가정 위반 |
| **M1b Ordered Logistic** | 서열변수 | 비례 오즈(proportional odds) | 서열변수에 이론적으로 정합 | 계수 해석 복잡, 비례 오즈 가정 검증 필요 |
| **M1c Binary Logistic** | 이분 (A 이상=1 vs B+ 이하=0) | 로짓 | 강건, 해석 쉬움, 그룹 비율 적정(138:243) | 중간 등급 정보 손실 |

**채택: 3종 모두 실행 후 결과 비교 (Robustness Check)**
- 단일 모형의 한계를 다른 모형이 보완
- 3종 모두에서 방향과 유의성이 일관되면 결과의 견고성 입증
- OLS는 직관적 baseline, Ordered는 서열 정합, Binary는 강건성 확인

**비채택 대안**
- OLS 단독: 등간성 가정 위반 — Ordered와 Binary로 robustness 확인 필요
- Ordered 단독: 비례 오즈 가정 검증 필요, 계수 해석 어려움
- 분류 모형(Random Forest 등): 예측이 목적이 아니라 연관성 해석이 목적 — 부적합

### **변수 구성**

**종속변수**
- OLS·Ordered: `esg_grade_num` (0~6)
- Binary: `is_high` = (esg_grade_num ≥ 4, A 이상=1)

**독립변수 — feature set 3종 분리 (다중공선성 Decision Box ⑮)**
- M1a: seed_score_E/S/G + log_n_tokens
- M1b: expanded_score_E/S/G + log_n_tokens
- M1c: ref_cosine_E/S/G + log_n_tokens

**통제변수**
- `log_n_tokens`: cheap-talk 통제 (4장에서 ρ=0.663 확인, 필수)
- 연도·업종 FE는 6장 알파에서 추가

### **Decision Box ⑱ — 표준화 처리**

**표준화 없음 (원본 단위 유지)**
- 목적이 변수 간 효과크기 비교가 아닌 cheap-talk 통제 후 방향 확인
- `log_n_tokens` 계수를 원본 단위로 해석하는 것이 cheap-talk 논의에 직관적
- 표준화 시 `log_n_tokens` vs ESG feature 계수 비교는 6장 알파에서 별도 확인

In [ ]:
"""
5. 회귀 3종

M1a OLS      : esg_grade_num ~ feature_set + log_n_tokens
M1b Ordered  : esg_grade_num (서열) ~ feature_set + log_n_tokens
M1c Binary   : is_high(A이상) ~ feature_set + log_n_tokens

feature_set 3종: seed / expanded / cosine
"""
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

ANALYSIS_DF["is_high"] = (ANALYSIS_DF["esg_grade_num"] >= 4).astype(int)
y_ols    = ANALYSIS_DF["esg_grade_num"]
y_bin    = ANALYSIS_DF["is_high"]
print(f"[Binary 종속변수] A 이상: {y_bin.sum()} ({y_bin.mean()*100:.1f}%)")

feature_sets = {
    "seed":     ["seed_score_E",     "seed_score_S",     "seed_score_G"],
    "expanded": ["expanded_score_E", "expanded_score_S", "expanded_score_G"],
    "cosine":   ["ref_cosine_E",     "ref_cosine_S",     "ref_cosine_G"],
}

ols_res     = {}
ordered_res = {}
binary_res  = {}

for fs_name, feats in feature_sets.items():
    X_cols = feats + ["log_n_tokens"]
    X      = ANALYSIS_DF[X_cols].copy()

    # ── M1a OLS ──────────────────────────────────────────────────
    m_ols = sm.OLS(y_ols, sm.add_constant(X)).fit()
    ols_res[fs_name] = m_ols

    # ── M1b Ordered ──────────────────────────────────────────────
    m_ord = OrderedModel(y_ols, X, distr="logit").fit(method="bfgs", disp=False)
    ordered_res[fs_name] = m_ord

    # ── M1c Binary ───────────────────────────────────────────────
    m_bin = sm.Logit(y_bin, sm.add_constant(X)).fit(disp=False)
    binary_res[fs_name] = m_bin

# ── 결과 출력 ─────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("[M1a OLS — Y: esg_grade_num]")
for fs, m in ols_res.items():
    print(f"\n  --- {fs} ---  R²={m.rsquared:.3f}  Adj R²={m.rsquared_adj:.3f}  N={int(m.nobs)}")
    coef = pd.DataFrame({
        "coef": m.params, "p": m.pvalues,
        "ci_lo": m.conf_int()[0], "ci_hi": m.conf_int()[1],
    }).round(4)
    print(coef.to_string())

print(f"\n{'='*65}")
print("[M1b Ordered Logistic — Y: esg_grade_num (서열)]")
for fs, m in ordered_res.items():
    n_feats = len(feature_sets[fs]) + 1
    print(f"\n  --- {fs} ---  LogLik={m.llf:.1f}  LR p={m.llr_pvalue:.4f}")
    coef = pd.DataFrame({
        "coef": m.params[:n_feats],
        "p":    m.pvalues[:n_feats],
    }, index=feature_sets[fs]+["log_n_tokens"]).round(4)
    print(coef.to_string())

print(f"\n{'='*65}")
print("[M1c Binary Logistic — Y: A 이상(1) vs B+ 이하(0)]")
for fs, m in binary_res.items():
    print(f"\n  --- {fs} ---  PseudoR²={m.prsquared:.3f}  LR p={m.llr_pvalue:.6f}")
    coef = pd.DataFrame({
        "coef": m.params,
        "OR":   np.exp(m.params),
        "p":    m.pvalues,
    }).round(4)
    print(coef.to_string())

# ── 3종 종합 비교 — log_n_tokens 효과 ─────────────────────────────
print(f"\n{'='*65}")
print("[3종 모형 종합 비교 — log_n_tokens 계수]")
summary = []
for fs in feature_sets:
    summary.append({
        "feature_set":    fs,
        "OLS_R²":         round(ols_res[fs].rsquared, 3),
        "OLS_lnt_β":      round(ols_res[fs].params["log_n_tokens"], 3),
        "OLS_lnt_p":      round(ols_res[fs].pvalues["log_n_tokens"], 4),
        "Ord_lnt_β":      round(ordered_res[fs].params["log_n_tokens"], 3),
        "Ord_lnt_p":      round(ordered_res[fs].pvalues["log_n_tokens"], 4),
        "Bin_PseudoR²":   round(binary_res[fs].prsquared, 3),
        "Bin_lnt_OR":     round(np.exp(binary_res[fs].params["log_n_tokens"]), 3),
        "Bin_lnt_p":      round(binary_res[fs].pvalues["log_n_tokens"], 4),
    })
print(pd.DataFrame(summary).to_string(index=False))

# 저장
pd.DataFrame(summary).to_csv(
    os.path.join(OUTPUT_DIR,"regression_summary.csv"),
    index=False, encoding="utf-8-sig"
)
print(f"\n[저장] regression_summary.csv")

[Binary 종속변수] A 이상: 138 (36.2%)

[M1a OLS — Y: esg_grade_num]

  --- seed ---  R²=0.415  Adj R²=0.409  N=381
                coef       p    ci_lo    ci_hi
const        -9.2837  0.0000 -10.8560  -7.7114
seed_score_E  1.0398  0.0812  -0.1294   2.2089
seed_score_S -1.3399  0.3619  -4.2260   1.5463
seed_score_G  6.6345  0.0669  -0.4639  13.7328
log_n_tokens  1.4152  0.0000   1.2141   1.6162

  --- expanded ---  R²=0.476  Adj R²=0.471  N=381
                    coef       p    ci_lo   ci_hi
const            -8.9143  0.0000 -10.4330 -7.3956
expanded_score_E  0.4322  0.0254   0.0535  0.8110
expanded_score_S -0.6949  0.3024  -2.0179  0.6281
expanded_score_G  3.1735  0.0000   2.2586  4.0885
log_n_tokens      1.2967  0.0000   1.1009  1.4926

  --- cosine ---  R²=0.429  Adj R²=0.423  N=381
                 coef       p    ci_lo    ci_hi
const         -9.2803  0.0000 -10.8117  -7.7489
ref_cosine_E   2.3136  0.3251  -2.3032   6.9303
ref_cosine_S   4.8486  0.2135  -2.8014  12.4986
ref_cosine_G  13.

**회귀 3종 완료**

**M1a OLS 결과 요약**

| feature set | R² | log_n_tokens β | p | 해석 |
|---|---|---|---|---|
| seed | 0.415 | 1.415 | <0.001 | |
| **expanded** | **0.476** | **1.297** | **<0.001** | **3종 중 R² 최고** |
| cosine | 0.429 | 1.379 | <0.001 | |

**ESG feature 계수 패턴 (OLS)**

| feature | β | p | 해석 |
|---|---|---|---|
| expanded_score_G | **3.174** | <0.001 | 3종 feature set 통틀어 유일하게 강하게 유의 |
| expanded_score_E | 0.432 | 0.025 | 약하게 유의 |
| ref_cosine_G | 13.837 | <0.001 | 유의하나 단위 차이로 계수 크게 보임 |
| seed_score_G | 6.635 | 0.067 | 경계선 유의 (p=0.067) |
| **expanded_score_S** | **−0.695** | **0.302** | **비유의, 부호 음수** |
| seed_score_S | −1.340 | 0.362 | 비유의, 부호 음수 |

**M1b Ordered Logistic 결과**

- `log_n_tokens` β ≈ 2.0 — 3종 feature set 모두에서 일관, p<0.001
- `expanded_score_G` β=4.944, p<0.001 — OLS와 방향 일치
- `expanded_score_E` β=0.551, p=0.054 — 경계선 유의

**M1c Binary Logistic 결과**

| feature set | PseudoR² | log_n_tokens OR | p |
|---|---|---|---|
| seed | 0.248 | 7.656 | <0.001 |
| **expanded** | **0.270** | **7.857** | **<0.001** |
| cosine | 0.243 | 8.042 | <0.001 |

- **log_n_tokens OR ≈ 7.9**: 토큰 수가 e배(약 2.7배) 증가 시 A 이상 등급일 odds 약 8배 증가
- `expanded_score_G` OR=27.1, p<0.001 — 가장 강하고 일관된 ESG 신호
- `seed_score_G` OR=9,842,210, p=0.031 — 계수 추정 불안정 (seed_G 분포 극단적 희소)

**3종 모형 종합 비교 — log_n_tokens 효과**

| feature set | OLS R² | OLS β | Ord β | Bin OR | Bin PseudoR² |
|---|---|---|---|---|---|
| seed | 0.415 | 1.415 | 2.055 | 7.656 | 0.248 |
| **expanded** | **0.476** | **1.297** | **2.027** | **7.857** | **0.270** |
| cosine | 0.429 | 1.379 | 2.041 | 8.042 | 0.243 |

**결론**

1. **log_n_tokens가 3종 모형 모두에서 압도적으로 유의** (p<0.001 일관) — cheap-talk 가설 회귀 수준에서 확증
2. **expanded_score_G가 가장 견고한 ESG 신호** — OLS·Ordered·Binary 3종 모두 유의 (p<0.001)
3. **S 차원 계수가 모든 모형에서 음수(비유의)** — 사업보고서 S 어휘가 등급과 역방향 또는 무관. 공시 내용이 S 성과를 측정하지 못함
4. **E 차원은 expanded에서만 약하게 유의** — seed만으로는 E 신호도 불안정
5. **R² 41~48% 중 대부분이 log_n_tokens 설명** — ESG feature 추가 설명력은 제한적

---

# **6. 알파 분석 — Cheap-talk 정면 검토 · Robustness**

## **6-1. 알파 설계**

5단계 결과에서 도출된 핵심 미해결 질문 4개를 알파로 설계한다.

| 알파 | 질문 | 방법 |
|---|---|---|
| **α1. Verbosity 직교화** | ESG feature가 분량의 대리지표인가, 독립적 신호인가? | log_n_tokens로 feature 회귀 → 잔차로 등급 회귀 |
| **α2. Industry·Year FE** | 업종·연도 효과를 통제해도 결과가 유지되는가? | 더미변수 추가 후 계수 변화 확인 |
| **α3. Section-level** | II·IV·VI 중 어느 섹션이 ESG 신호를 더 많이 담는가? | 섹션별 글자 수로 등급 회귀 |
| **α4. Governance Paradox** | G 의무공시 어휘와 G 실천 어휘는 왜 다른 패턴을 보이는가? | seed_G vs expanded_G 분포 비교 |

**본 분석 vs 알파 구분**
- 본 분석(5단계): OLS·Ordered·Binary, 381 전수, log_n_tokens 통제
- 알파(6단계): 직교화·FE·Section·Paradox — 본선 결과의 robustness check

In [ ]:
"""
6. 알파 분석 4종

α1. Verbosity 직교화
α2. Industry·Year FE
α3. Section-level 회귀
α4. Governance Paradox
"""

# ── α1. Verbosity 직교화 ──────────────────────────────────────────
print("=" * 60)
print("[α1. Verbosity 직교화]")
print("  ESG feature를 log_n_tokens로 회귀 → 잔차 → 잔차로 등급 회귀")

focus = ["expanded_score_E", "expanded_score_S", "expanded_score_G"]
resid_df = pd.DataFrame(index=ANALYSIS_DF.index)

print(f"\n  [Step 1] log_n_tokens가 각 feature를 얼마나 설명하는가")
for feat in focus:
    m = sm.OLS(ANALYSIS_DF[feat],
               sm.add_constant(ANALYSIS_DF[["log_n_tokens"]])).fit()
    resid_df[f"{feat}_resid"] = m.resid
    print(f"    {feat}: R²={m.rsquared:.3f} "
          f"→ log_n_tokens가 {m.rsquared*100:.1f}% 설명")

print(f"\n  [Step 2] 잔차 + log_n_tokens → esg_grade_num")
X_orth = sm.add_constant(pd.concat([
    resid_df[[f"{f}_resid" for f in focus]],
    ANALYSIS_DF[["log_n_tokens"]],
], axis=1))
m_orth = sm.OLS(ANALYSIS_DF["esg_grade_num"], X_orth).fit()
print(f"    R²={m_orth.rsquared:.3f}")
for feat in focus:
    r = f"{feat}_resid"
    b = m_orth.params[r]
    p = m_orth.pvalues[r]
    sig = "★유의" if p < 0.05 else "비유의"
    print(f"    {r}: β={b:.3f} (p={p:.4f}) → {sig}")

n_sig = sum(1 for f in focus if m_orth.pvalues[f"{f}_resid"] < 0.05)
print(f"\n  → 직교화 후 유의 feature: {n_sig}/{len(focus)}개")
if n_sig == 0:
    print(f"     ESG feature는 분량의 대리지표 — cheap-talk 결정적 증거")
elif n_sig < len(focus):
    print(f"     일부 feature만 독립 신호 보유 — cheap-talk 부분 확증")
else:
    print(f"     모든 feature 독립 신호 — cheap-talk만으론 설명 안 됨")

# ── α2. Industry·Year FE ──────────────────────────────────────────
print(f"\n{'='*60}")
print("[α2. Industry·Year FE]")

# industry 정보
ind_df = META_INPUT_DF[["stock_code","fiscal_year","industry"]].copy()
ind_df["stock_code"]  = ind_df["stock_code"].str.zfill(6)
ind_df["fiscal_year"] = ind_df["fiscal_year"].astype(int)
ADF = ANALYSIS_DF.merge(ind_df, on=["stock_code","fiscal_year"], how="left")

# 소규모 업종 통합
ind_counts = ADF["industry"].value_counts()
ADF["industry_fe"] = ADF["industry"].apply(
    lambda x: x if ind_counts.get(x, 0) >= 5 else "기타"
)
print(f"  업종 그룹: {ADF['industry_fe'].nunique()}개")

# expanded feature set 기준 FE 회귀
X_fe = pd.concat([
    ADF[["expanded_score_E","expanded_score_S","expanded_score_G","log_n_tokens"]],
    pd.get_dummies(ADF["industry_fe"], drop_first=True, dtype=float),
    pd.get_dummies(ADF["fiscal_year"],  drop_first=True, dtype=float),
], axis=1)
m_fe = sm.OLS(ADF["esg_grade_num"], sm.add_constant(X_fe)).fit()

print(f"\n  본선 (expanded, FE 없음):  R²={ols_res['expanded'].rsquared:.3f}  "
      f"log_n_tokens β={ols_res['expanded'].params['log_n_tokens']:.3f}")
print(f"  FE 추가 (industry+year): R²={m_fe.rsquared:.3f}  "
      f"log_n_tokens β={m_fe.params['log_n_tokens']:.3f}")
change = (m_fe.params["log_n_tokens"] -
          ols_res["expanded"].params["log_n_tokens"])
print(f"  → log_n_tokens β 변화: {change:+.3f} "
      f"({abs(change)/ols_res['expanded'].params['log_n_tokens']*100:.1f}%)")
for feat in ["expanded_score_E","expanded_score_S","expanded_score_G"]:
    b = m_fe.params[feat]
    p = m_fe.pvalues[feat]
    print(f"    {feat}: β={b:.3f} (p={p:.4f})")

# ── α3. Section-level ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("[α3. Section-level 회귀]")

# META_DF에서 섹션 글자 수 join
sec_cols = ["stock_code","fiscal_year",
            "section_chars_II","section_chars_IV","section_chars_VI"]
avail = [c for c in sec_cols if c in META_DF.columns]
if len(avail) == len(sec_cols):
    sec_df = META_DF[avail].copy()
    sec_df["stock_code"]  = sec_df["stock_code"].fillna("").str.zfill(6)
    sec_df["fiscal_year"] = sec_df["fiscal_year"].astype(int)
    ADF2 = ANALYSIS_DF.merge(sec_df, on=["stock_code","fiscal_year"], how="left")
    for c in ["section_chars_II","section_chars_IV","section_chars_VI"]:
        ADF2[f"log_{c}"] = np.log1p(ADF2[c])

    X_sec = sm.add_constant(
        ADF2[["log_section_chars_II","log_section_chars_IV","log_section_chars_VI"]]
    )
    m_sec = sm.OLS(ADF2["esg_grade_num"], X_sec).fit()
    print(f"  R²={m_sec.rsquared:.3f}")
    for c in ["log_section_chars_II","log_section_chars_IV","log_section_chars_VI"]:
        b = m_sec.params[c]
        p = m_sec.pvalues[c]
        sig = "★" if p < 0.05 else ""
        print(f"  {c}: β={b:.3f} (p={p:.4f}) {sig}")
else:
    print(f"  ⚠ section_chars 컬럼 없음 — corpus JSON에서 보완 필요")
    # corpus JSON에서 섹션 글자 수 재구성
    sec_records = []
    for fpath in sorted(glob.glob(os.path.join(CORPUS_DIR,"*.json"))):
        with open(fpath,"r",encoding="utf-8") as f:
            doc = json.load(f)
        sec_records.append({
            "stock_code":       doc["stock_code"],
            "fiscal_year":      int(doc["fiscal_year"]),
            "section_chars_II": doc.get("section_chars_II",0),
            "section_chars_IV": doc.get("section_chars_IV",0),
            "section_chars_VI": doc.get("section_chars_VI",0),
        })
    sec_df = pd.DataFrame(sec_records)
    sec_df["stock_code"] = sec_df["stock_code"].str.zfill(6)
    ADF2 = ANALYSIS_DF.merge(sec_df, on=["stock_code","fiscal_year"], how="left")
    for c in ["section_chars_II","section_chars_IV","section_chars_VI"]:
        ADF2[f"log_{c}"] = np.log1p(ADF2[c])

    X_sec = sm.add_constant(
        ADF2[["log_section_chars_II","log_section_chars_IV","log_section_chars_VI"]]
    )
    m_sec = sm.OLS(ADF2["esg_grade_num"], X_sec).fit()
    print(f"  R²={m_sec.rsquared:.3f}")
    for c in ["log_section_chars_II","log_section_chars_IV","log_section_chars_VI"]:
        b = m_sec.params[c]
        p = m_sec.pvalues[c]
        sig = "★" if p < 0.05 else ""
        print(f"  {c}: β={b:.3f} (p={p:.4f}) {sig}")

# ── α4. Governance Paradox ────────────────────────────────────────
print(f"\n{'='*60}")
print("[α4. Governance Paradox]")
print("  seed_G vs expanded_G 분포 비교")

for col, label in [("seed_score_G","seed_G"),("expanded_score_G","expanded_G")]:
    vals = ANALYSIS_DF[col]
    cv   = vals.std() / vals.mean() if vals.mean() > 0 else float("inf")
    n_zero = (vals == 0).sum()
    print(f"\n  [{label}]")
    print(f"    평균: {vals.mean():.4f}  std: {vals.std():.4f}  CV: {cv:.2f}")
    print(f"    0점 firm-year: {n_zero}건 ({n_zero/len(vals)*100:.1f}%)")

print(f"\n  [expanded_G → g_grade_num OLS]")
X_g = sm.add_constant(
    ANALYSIS_DF[["expanded_score_G","log_n_tokens"]]
)
m_g = sm.OLS(ANALYSIS_DF["g_grade_num"], X_g).fit()
print(f"    R²={m_g.rsquared:.3f}")
print(f"    expanded_G β={m_g.params['expanded_score_G']:.3f} "
      f"(p={m_g.pvalues['expanded_score_G']:.4f})")
print(f"    log_n_tokens β={m_g.params['log_n_tokens']:.3f} "
      f"(p={m_g.pvalues['log_n_tokens']:.4f})")

[α1. Verbosity 직교화]
  ESG feature를 log_n_tokens로 회귀 → 잔차 → 잔차로 등급 회귀

  [Step 1] log_n_tokens가 각 feature를 얼마나 설명하는가
    expanded_score_E: R²=0.178 → log_n_tokens가 17.8% 설명
    expanded_score_S: R²=0.113 → log_n_tokens가 11.3% 설명
    expanded_score_G: R²=0.032 → log_n_tokens가 3.2% 설명

  [Step 2] 잔차 + log_n_tokens → esg_grade_num
    R²=0.476
    expanded_score_E_resid: β=0.432 (p=0.0254) → ★유의
    expanded_score_S_resid: β=-0.695 (p=0.3024) → 비유의
    expanded_score_G_resid: β=3.174 (p=0.0000) → ★유의

  → 직교화 후 유의 feature: 2/3개
     일부 feature만 독립 신호 보유 — cheap-talk 부분 확증

[α2. Industry·Year FE]
  업종 그룹: 4개

  본선 (expanded, FE 없음):  R²=0.476  log_n_tokens β=1.297
  FE 추가 (industry+year): R²=0.490  log_n_tokens β=1.311
  → log_n_tokens β 변화: +0.014 (1.1%)
    expanded_score_E: β=0.401 (p=0.0462)
    expanded_score_S: β=-0.733 (p=0.2760)
    expanded_score_G: β=3.122 (p=0.0000)

[α3. Section-level 회귀]
  R²=0.431
  log_section_chars_II: β=0.355 (p=0.0001) ★
  log_section_chars_IV: β=0.726 (p=

**알파 분석 4종 완료**

**α1. Verbosity 직교화**

**알파 분석 채택 이유**

5. 회귀분석에서 `log_n_tokens`가 ESG feature보다 더 강한 효과를 보였다.  
이때 ESG feature 계수가 양수인 것이 진짜 ESG 신호인지, ESG feature 자체가 분량과 강하게 상관되어 있어 그 잔여 효과만 잡힌 것인지 구분할 필요가 있다.  
두 변수를 같은 회귀에 그냥 넣으면 다중공선성으로 계수 해석이 흔들리므로 ESG feature에서 분량으로 설명되는 부분을 먼저 분리해야 한다.  
이것이 Frisch-Waugh 정리에 기반한 직교화이다.

**설계**

- Step 1: 각 ESG feature를 `log_n_tokens`로 단순 회귀 → 잔차 `r_i` 추출
  - 잔차 = "분량으로 설명되지 않는 ESG feature 부분"
  - R²가 높을수록 그 feature가 분량의 대리지표에 가깝다는 뜻
- Step 2: 잔차 `r_i`와 `log_n_tokens`를 함께 `esg_grade_num`에 회귀
- 잔차 계수가 유의하면 → 분량과 독립된 ESG 신호 보유
- 잔차 계수가 0에 가까우면 → 그 feature는 사실상 분량의 대리지표

**결과**

| feature | log_n_tokens 설명력(R²) | 직교화 후 β | p | 판정 |
|---|---|---|---|---|
| expanded_score_E | 17.8% | 0.432 | 0.025 | **★ 독립 신호 보유** |
| expanded_score_S | 11.3% | −0.695 | 0.302 | 비유의 — 분량 대리지표 |
| expanded_score_G | 3.2% | 3.174 | <0.001 | **★ 독립 신호 보유** |

- **expanded_G는 log_n_tokens와 거의 무관(R²=3.2%)하면서도 등급과 강하게 연관** — G 차원의 ESG 신호가 분량 효과와 독립적임을 가장 명확히 입증
- expanded_E는 분량의 17.8%를 공유하지만 직교화 후에도 유의 — 부분적 독립 신호
- **expanded_S는 직교화 후 완전히 비유의** — S feature는 사실상 분량의 대리지표
- 결과적으로 "분량이 지배적이지만, G·E에는 분량과 독립적인 약한 ESG 신호가 존재한다"는 본 분석의 핵심 주장에 정량 근거 제공

**한계**

- OLS의 수학적 성질상 직교화 회귀의 전체 R²는 본선과 동일 — 본 알파는 개별 계수의 분해에만 의미가 있고 모형 전체 설명력 개선이 아님
- 직교화는 "분량을 통제했을 때의 ESG 효과"를 보지만 cheap-talk을 반증하지는 못함 — 직교화 후에도 유의한 G·E 신호가 진짜 ESG 성과 차이인지 다른 미관측 요인(예: 보고서 작성 관행 차이)인지는 여전히 식별 불가
- `log_n_tokens` 하나로만 통제 — 평균 문장 길이, 어휘 다양성 같은 verbosity의 다른 차원은 미통제

**α2. Industry·Year FE**

**알파 분석 채택 이유**

5. 회귀분석에서 `log_n_tokens`의 효과가 압도적으로 강했다. 이 효과가 정말 분량 자체의 영향인지, 특정 업종이나 특정 연도가 더 길게 쓰는 경향이 있는데 그 업종·연도가 우연히 등급도 높은 것인지 구분해야 cheap-talk 결론을 유지할 수 있다. 업종·연도 더미를 통제했을 때 `log_n_tokens` 계수가 그대로면 → cheap-talk은 업종/연도 부산물이 아니다.

**설계**

- 업종 = `industry` (KCGS 분류), 5건 미만 업종은 "기타"로 통합 → 4그룹 (전기전자·금융·화학·기타)
- 연도 더미 = 2022·2023·2024
- 5단계 OLS(`expanded` feature set + `log_n_tokens`)에 업종·연도 더미만 추가
- 비교 지표: `log_n_tokens` β의 변화율, expanded_E/S/G 계수의 유의성 유지 여부

**결과**

| 모형 | R² | log_n_tokens β | β 변화 |
|---|---|---|---|
| 본선 (FE 없음) | 0.476 | 1.297 | — |
| FE 추가 (업종+연도) | 0.490 | 1.311 | **+0.014 (1.1%)** |

- FE 추가 후 `log_n_tokens` β 변화 1.1% — **cheap-talk 효과가 업종·연도 효과와 독립적**
- expanded_G β=3.122(p<0.001) — FE 통제 후에도 견고하게 유의
- expanded_E β=0.401(p=0.046) — FE 후에도 약하게 유의 유지
- expanded_S는 본선과 동일하게 비유의 — S 차원 결론도 견고

**한계**

- 업종 그룹이 4개로 적음 — 진정한 산업 효과를 완전 통제하려면 KCGS 분류 그대로(15+ 업종) 사용해야 하나, 표본 N=381에서 셀당 추정량이 부족해 통합 불가피
- 연도 3개로 시계열 패턴 식별 약함 — 2024년에 cheap-talk이 더 강해졌는가같은 시기별 진단 불가
- 본 알파는 cheap-talk 견고성을 보일 뿐 cheap-talk 자체를 반증하지는 못함

**α3. Section-level 회귀**

**알파 분석 채택 이유**

본 분석은 II/IV/VI 세 섹션을 통합해 firm-year 1행으로 처리했다.  
그러나 각 섹션은 성격이 다르다 — II(사업의 내용)는 사업·재무 서술 중심, IV(경영진단)는 경영진의 자발적 ESG 서술, VI(이사회)는 G 관련 의무 공시  
만약 cheap-talk이 어느 한 섹션 분량 때문에 발생한다면 그 섹션을 식별하는 것이 ESG 신호의 발화 지점 진단에 결정적이다.  
또한 어느 섹션의 글자 수가 등급과 가장 강하게 연관되는가는 의무 공시(법정) vs 자발적 공시 구분과 직접 맞닿아 있어 정책 함의도 다르다.

**설계**

- 각 firm-year의 섹션별 글자 수 `section_chars_II/IV/VI`를 `collection_meta.csv`에서 가져옴
- 로그 변환 (`log1p`) — 우측 꼬리 분포 보정
- OLS: `esg_grade_num ~ log_section_chars_II + log_section_chars_IV + log_section_chars_VI`
- 본선의 `log_n_tokens` 단독 회귀와 비교 — 3개 섹션 분량의 합 vs 각각 분리의 효과

**결과**

| 섹션 | β | p | 해석 |
|---|---|---|---|
| log_section_chars_II (사업의 내용) | 0.355 | <0.001 ★ | 가장 약함 — 재무·사업 위주 |
| **log_section_chars_IV (경영진단)** | **0.726** | **<0.001 ★** | **3개 섹션 중 최강** |
| log_section_chars_VI (이사회) | 0.701 | <0.001 ★ | G 서술 중심 |

- 3개 섹션 모두 유의 — R²=0.431, 총 글자 수 단독 회귀와 유사한 설명력
- IV(경영진단)가 가장 강한 효과 — 경영진이 자발적으로 ESG를 서술하는 섹션이 평가기관과 가장 강하게 연관
- VI(이사회)도 비슷한 효과 — G 차원 의무 공시의 분량이 KCGS 등급과 연관
- II β가 가장 낮음 — 재무·사업 위주 서술이 ESG 신호를 희석
- **함의**: cheap-talk은 단순 보고서 길이가 아니라 특정 섹션(특히 IV)의 길이가 주도 → "ESG 의지가 있는 기업은 경영진단 섹션에서 더 길게 쓴다" 또는 "분량이 긴 기업은 IV에서 길게 써서 등급도 높다"로 양방향 해석 가능

**한계**

- 섹션 분량과 ESG feature(expanded_E/S/G)를 동시 투입할 경우 다중공선성 — 본 알파에서는 섹션 분량만 단독 회귀, ESG feature 효과와의 상호작용 분리 불가
- 섹션별 ESG 어휘 강도(`expanded_score_E_section_IV` 같은 분해)는 본 알파 범위 밖 — 이를 분리하면 cheap-talk 원천을 더 정밀하게 식별할 수 있으나 firm-year × 섹션 × 차원의 9개 feature 추가가 필요해 본 분석에서는 단순화
- IV가 강한 이유가 ESG 의지 때문인지 MD&A 작성 관행 때문인지 본 분석에서는 식별 불가

**α4. Governance Paradox**

**알파 분석 채택 이유**

3. TF-IDF 결과에서 G seed 30개 중 7개(`이사회`, `사외이사`, `주주`, `감사위원회`, `독립성`, `준법`, `의결권`)가 `max_df=0.80`에 걸려 vocab에서 자동 제외됐다.  
거의 모든 firm-year에 등장(80% 이상)해 변별력이 없기 때문이다.  
그러나 이 단어들은 상장기업 법정 의무 공시에 들어가는 어휘이고, 사실상 모든 기업이 비슷한 표현을 쓰기 때문에 등급 차이를 만들지 못한다.  
반면 expanded dictionary로 확장한 G 어휘(감사위원, 부패방지, 내부통제 등)는 firm-year별로 실제 차별적 사용 패턴을 보인다.  
이 의무 공시 어휘 vs 실천 어휘의 분리를 정량 입증하는 것이 본 알파의 목적이고, 이는 seed-only 분석으로는 G 차원이 측정 불가에 가깝다는 결론으로 연결되어 expanded dictionary 채택 자체를 정당화한다.

**설계**

- 측정 1: 각 firm-year의 seed_G score와 expanded_G score 분포 비교
  - 평균, 표준편차, 변동계수 CV (= std/mean), 0점 firm-year 비율
- 측정 2: 두 점수가 G grade를 얼마나 설명하는지 단순 OLS
  - seed_G → g_grade vs expanded_G → g_grade
- 핵심 진단: 0점 firm-year 비율 — 어휘가 대부분 firm에 똑같이 등장하면 max_df 적용 후 0점이 다수 발생

**결과**

| 측정 | 평균 | std | CV | 0점 firm-year |
|---|---|---|---|---|
| seed_G | 0.0086 | 0.0180 | **2.09** | **281건 (73.8%)** |
| expanded_G | 0.1965 | 0.1333 | **0.68** | 9건 (2.4%) |

- **seed_G: 381건 중 281건(73.8%)이 0점** — 의무 공시 어휘는 max_df 제외 후 측정 자체가 불가능. CV=2.09는 비정상적으로 큰 변동, 즉 *0점이 압도적이고 일부 firm만 비0점*인 상태
- **expanded_G: 2.4%만 0점, CV=0.68** — 감사위원·부패·내부통제 등 실천 어휘는 firm-year별로 차별적으로 사용
- expanded_G → g_grade OLS: β=2.812(p<0.001), R²=0.414 — expanded로 보강한 G 신호는 g_grade를 41% 설명
- 의무공시 어휘(seed_G) ≠ 실천 어휘(expanded_G)의 분리가 수치로 명확히 입증
- 이 결과는 단순히 G 차원의 한 발견이 아니라 expanded dictionary 채택의 핵심 정당성 — G에서 expanded 없이는 측정 불가

**한계**

- expanded_G의 차별적 사용이 진짜 지배구조 실천의 차이인지 공시 작성 스타일의 차이인지 식별 불가 — 어휘 사용 ≠ 실제 실천
- expanded_G 사전 자체가 큐레이션의 산물 — 채택/기각 판단에 주관성 존재
- 본 알파는 G 차원에서만 진단 — E·S에도 유사한 의무 공시 vs 실천 어휘 분리가 존재할 가능성이 있으나 본 분석 범위 밖
- `max_df=0.80` 임계값 자체가 선택의 산물 — 임계값을 바꾸면 어느 어휘가 의무 공시로 분류되는지 달라질 수 있음

**종합**

| 알파 | 핵심 질문 | 본선과 일관성 | 새로운 발견 | 한계 |
|---|---|---|---|---|
| α1 직교화 | ESG feature가 분량과 독립적인가? | 일관 | E·G만 독립 신호, S는 분량 대리지표 | OLS 성질상 본선과 동일 R², cheap-talk 반증 아님 |
| α2 FE | cheap-talk이 업종·연도 부산물인가? | **견고** — β 변화 1.1% | 업종·연도 통제 후에도 cheap-talk 유지 | 업종 4그룹 단순화, 시계열 약함 |
| α3 Section | 어느 섹션이 cheap-talk을 주도하는가? | 추가 인사이트 | IV(경영진단) β=0.726이 최강 — 자발적 서술 섹션 | 다중공선성 회피 위해 ESG feature와 분리, IV 강함의 원인 식별 불가 |
| α4 Paradox | 의무 공시 어휘 vs 실천 어휘? | 일관 | seed_G 73.8%가 0점 — 의무 공시 측정 한계 입증 | expanded_G 큐레이션 주관성, G 차원에만 진단 |

---

# **7. Cheap-talk Evidence Matrix**

## **7-1. 목적**

모든 분석 결과를 cheap-talk 가설의 증거로 통합한다.

**Cheap-talk 가설**
> KCGS ESG 등급은 사업보고서의 'ESG 어휘 진정성'보다 '공시 분량'을 더 강하게 반영한다.

이 가설을 지지하는 증거와 반증하는 증거를 함께 제시하고, 가설의 견고성을 종합 평가한다.

In [ ]:
"""
7. Cheap-talk Evidence Matrix
모든 분석 결과를 증거 카테고리별로 통합
"""

evidence = [
    # (카테고리, 발견, 수치, 출처, 결론)
    ("1.분량 압도",
     "n_tokens ↔ esg_grade Spearman ρ",
     0.663, "4장",
     "분량이 어떤 ESG feature보다 강함 (최강 ESG: 0.451)"),
    ("1.분량 압도",
     "n_tokens Mann-Whitney 효과크기",
     0.649, "4장",
     "A이상 평균 7,523토큰 vs B+이하 3,547토큰 — 2.1배 차이"),
    ("2.모형 무관 견고",
     "OLS log_n_tokens β (expanded)",
     1.297, "5장",
     "p<0.001, 3종 feature set 모두 일관"),
    ("2.모형 무관 견고",
     "Ordered log_n_tokens β",
     2.027, "5장",
     "p<0.001"),
    ("2.모형 무관 견고",
     "Binary log_n_tokens OR",
     7.857, "5장",
     "A이상 odds 7.9배 — 토큰 수 e배 증가 시"),
    ("3.FE 후 견고",
     "Industry+Year FE 후 log_n_tokens β",
     1.311, "6장α2",
     "본선 1.297 대비 변화 1.1% — 업종·연도 무관"),
    ("4.차원별 비대칭",
     "expanded_S 계수 부호",
     -0.695, "5장",
     "모든 모형에서 음수(비유의) — S 어휘 측정 한계"),
    ("4.차원별 비대칭",
     "expanded_G β (가장 강한 ESG 신호)",
     3.174, "5장",
     "3종 모형 모두 p<0.001 — 유일하게 견고한 ESG 신호"),
    ("5.섹션별 차이",
     "log_section_chars_IV β",
     0.726, "6장α3",
     "3개 섹션 중 최강 — 자발적 ESG 서술 섹션"),
    ("5.섹션별 차이",
     "log_section_chars_II β",
     0.355, "6장α3",
     "가장 약함 — 사업·재무 서술 비중 높아 희석"),
    ("6.Governance Paradox",
     "seed_G 0점 비율",
     0.738, "6장α4",
     "73.8% firm-year에서 G seed 측정 불가 — 의무공시 어휘 변별력 없음"),
    ("6.Governance Paradox",
     "expanded_G CV",
     0.68, "6장α4",
     "실천 어휘는 차별적 사용 — seed_G CV=2.09 대비 정상 분포"),
    ("7.직교화",
     "직교화 후 expanded_G β",
     3.174, "6장α1",
     "log_n_tokens 분리 후에도 유의 — G는 독립 신호"),
    ("7.직교화",
     "직교화 후 expanded_S β",
     -0.695, "6장α1",
     "비유의 — S는 분량의 대리지표 확정"),
]

ev_df = pd.DataFrame(evidence,
    columns=["category","finding","value","source","conclusion"])

print("[Cheap-talk Evidence Matrix]")
for cat in ev_df["category"].unique():
    print(f"\n  ━━ {cat} ━━")
    sub = ev_df[ev_df["category"]==cat]
    for _, row in sub.iterrows():
        print(f"    [{row['source']}] {row['finding']}: {row['value']}")
        print(f"           → {row['conclusion']}")

ev_df.to_csv(os.path.join(OUTPUT_DIR,"cheaptalk_evidence_matrix.csv"),
             index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print("[Cheap-talk 가설 견고성 종합]")
print("""
  지지 증거 (강):
  - n_tokens가 모든 검증(Spearman·MWU·OLS·Ordered·Binary·FE)에서 압도적
  - FE 통제 후에도 β 변화 1.1% — 업종·연도 효과 아님
  - S 차원 계수 음수 — 어휘 풍부함이 S 등급과 무관 또는 역방향

  지지 증거 (약):
  - expanded_G는 직교화 후에도 유의 — G 어휘에 일부 독립 신호
  - expanded_E도 직교화 후 유의 — E 어휘에 부분적 독립 신호

  결론:
  "분량이 지배적이나, G·E 차원 실천 어휘에는 분량과 독립적인
   약한 ESG 신호가 존재한다. S 차원은 사업보고서로 측정 불가 수준."
""")
print(f"[저장] cheaptalk_evidence_matrix.csv")

[Cheap-talk Evidence Matrix — 최종 (α5·α6 추가)]
  총 18개 증거 / 9개 카테고리

  ━━ 1.분량 압도 ━━
    [4장] n_tokens ↔ esg_grade Spearman ρ: 0.663
           → 분량이 어떤 ESG feature보다 강함 (최강 ESG: 0.451)
    [4장] n_tokens Mann-Whitney 효과크기: 0.649
           → A이상 평균 7,523토큰 vs B+이하 3,547토큰 — 2.1배 차이

  ━━ 2.모형 무관 견고 ━━
    [5장] OLS log_n_tokens β (expanded): 1.297
           → p<0.001, 3종 feature set 모두 일관
    [5장] Ordered log_n_tokens β: 2.027
           → p<0.001
    [5장] Binary log_n_tokens OR: 7.857
           → A이상 odds 7.9배 — 토큰 수 e배 증가 시

  ━━ 3.FE 후 견고 ━━
    [6장α2] Industry+Year FE 후 log_n_tokens β: 1.311
           → 본선 1.297 대비 변화 1.1% — 업종·연도 무관

  ━━ 4.차원별 비대칭 ━━
    [5장] expanded_S 계수 부호: -0.695
           → 모든 모형에서 음수(비유의) — S 어휘 측정 한계
    [5장] expanded_G β (가장 강한 ESG 신호): 3.174
           → 3종 모형 모두 p<0.001 — 유일하게 견고한 ESG 신호

  ━━ 5.섹션별 차이 ━━
    [6장α3] log_section_chars_IV β: 0.726
           → 3개 섹션 중 최강 — 자발적 ESG 서술 섹션
    [6장α3] log_section_chars_II β: 0.355
           → 가장 약함 — 사업·재무 

**Cheap-talk Evidence Matrix 완료**

**7개 카테고리 × 14개 증거**

| 카테고리 | 핵심 수치 | 결론 |
|---|---|---|
| 1. 분량 압도 | n_tokens ρ=0.663, MWU r=0.649 | 분량이 모든 ESG feature 압도 |
| 2. 모형 무관 견고 | OLS β=1.297, Ordered β=2.027, Binary OR=7.857 | 3종 모형 모두 p<0.001 일관 |
| 3. FE 후 견고 | β 변화 1.1% | 업종·연도 효과 아님 |
| 4. 차원별 비대칭 | expanded_G β=3.174(p<0.001), expanded_S β=−0.695(비유의) | G만 견고, S는 측정 한계 |
| 5. 섹션별 차이 | IV β=0.726 > VI β=0.701 > II β=0.355 | 자발적 서술 섹션이 더 강함 |
| 6. Governance Paradox | seed_G 0점 73.8%, expanded_G 0점 2.4% | 의무공시 ≠ 실천 어휘 |
| 7. 직교화 | expanded_G 직교화 후 유의, expanded_S 비유의 | G 독립 신호, S 대리지표 확정 |

**종합 결론**

분량이 지배적이나, G·E 차원 실천 어휘에는 분량과 독립적인 약한 ESG 신호가 존재한다. S 차원은 사업보고서로 측정 불가 수준이다.

---

# **8. 종합 결론 · 한계 · Self-Check**

## **8-1. 연구 질문 답변**

> 사업보고서 텍스트에서 추출한 ESG 관련 어휘 강도가 KCGS ESG 등급과 통계적으로 연관되는가?  
> 이 연관성을 cheap-talk과 어떻게 구분할 수 있는가?

**답변 3가지**

1. **연관은 있으나 분량이 지배적이다.**  
   9개 ESG feature 모두 p<0.01로 등급과 통계적으로 연관되지만, 효과크기는 모두 약함~중간 수준이다. 반면 단순 토큰 수(n_tokens)는 ρ=0.663, Binary OR=7.857로 ESG feature를 압도한다.

2. **차원별 패턴이 다르다.**  
   G(지배구조) expanded 어휘는 직교화 후에도 독립적 신호를 보유하고, E(환경)는 부분적 독립 신호가 있다. S(사회)는 직교화 후 완전히 비유의로 사업보고서로는 측정이 어렵다.

3. **이는 인과가 아닌 연관 관찰이다.**  
   KCGS 평가 알고리즘과 사업보고서 텍스트가 어떻게 연결되는지를 보여주는 패턴이며, "ESG 어휘를 더 쓰면 등급이 오른다"는 인과 주장은 본 분석으로부터 할 수 없다.

## **8-2. 한계**

**데이터 한계**

| 한계 | 영향 | 처리 |
|---|---|---|
| 표본 N=381 firm-year | 업종×연도 셀별 추정 불안정 | 6. FE 알파로 확인 |
| 3개 연도 (2022–2024) | 시계열 패턴 제한적 | 연도 FE로만 통제 |
| 사업보고서 텍스트만 사용 | 지속가능경영보고서 등 미포함 | 가이드 명시 범위 준수 |
| KCGS 등급만 사용 | MSCI 등 다른 평가기관 비교 불가 | 가이드 명시 범위 준수 |

**방법론 한계**

| 한계 | 영향 | 처리 |
|---|---|---|
| max_df=0.80으로 G seed 7/10 vocab 제외 | seed_G 측정 불가 | expanded_G로 보완 |
| Expanded dictionary 큐레이션 주관성 | 일부 채택/기각 판단 | 명시적 기준 + 예시 표 |
| TF-IDF cosine 절대값 매우 낮음 | 보조 측정 한계 | 본선은 seed/expanded 사용 |
| OLS 등간성 가정 위반 | 서열변수 부적합 | Ordered·Binary로 보완 |
| 업종 FE 4그룹으로 단순화 | 산업 효과 부분 통제만 가능 | 알파 한계로 명시 |

**인과 추론 한계 (CRITICAL)**

**다음 주장은 본 분석으로부터 할 수 없다.**
- ❌ "ESG 어휘를 더 사용하면 KCGS 등급이 오른다"
- ❌ "사업보고서 분량을 늘리면 ESG 평가가 좋아진다"
- ❌ "G 어휘를 풍부히 쓰면 지배구조가 우수하다"

**다음 주장은 할 수 있다.**
- ✓ "이 표본에서 KCGS A 이상 등급 기업은 평균적으로 더 긴 사업보고서를 작성한다"
- ✓ "분량이 ESG 어휘 측정값보다 등급과 강하게 연관된다"
- ✓ "이 연관은 cheap-talk 가설과 일관된다"